# **KẾT NỐI GOOGLE DRIVE**

In [153]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [154]:
import os
print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', '.ipynb_checkpoints', 'Waiting', 'IS6002']


In [ ]:
folder_path = "/content/drive/MyDrive/IS6002"

for item in os.listdir(folder_path):
    full_path = os.path.join(folder_path, item)

    if os.path.isdir(full_path):
        print("[FOLDER]", item)
    else:
        print("[FILE]", item)

[FOLDER] merged_dataset


# **KIỂM TRA TẬP DATASET & QUÁ TRÌNH COMPILE SOLIDITY**

**Thư mục chứa Dataset**

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/IS6002/merged_dataset"

for item in os.listdir(DATASET_DIR):
    path = os.path.join(DATASET_DIR, item)

    if os.path.isdir(path):
        print("[FOLDER]", item)
    else:
        print("[FILE]", item)

[FILE] labels.csv
[FOLDER] contracts


**Kiểm tra tính đầy đủ của Dataset**

In [ ]:
from pathlib import Path
import pandas as pd

DATASET_DIR = Path(
    "/content/drive/MyDrive/IS6002/merged_dataset"
)

CONTRACTS_DIR = DATASET_DIR / "contracts"
LABELS_PATH = DATASET_DIR / "labels.csv"

sol_files = list(CONTRACTS_DIR.glob("*.sol"))
print("Số file Solidity:", len(sol_files))

df = pd.read_csv(LABELS_PATH)

print("Số dòng trong labels.csv:", len(df))
print("\nCác cột:")
print(df.columns.tolist())

print("\nPhân bố nhãn:")
print(df["label"].value_counts(dropna=False))

display(df.head())

Số file Solidity: 43232
Số dòng trong labels.csv: 43232

Các cột:
['contract_id', 'source_dataset', 'original_path', 'copied_path', 'vulnerability_type', 'label', 'sha256']

Phân bố nhãn:
label
Vulnerable        29861
Non-vulnerable    13371
Name: count, dtype: int64


,contract_id,source_dataset,original_path,copied_path,vulnerability_type,label,sha256
0,C000001,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000001.sol,Overflow-Underflow,Vulnerable,b9b6f263036d2c4d1bfc8a39d3185ccb2a2d620ade2b90...
1,C000002,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000002.sol,Overflow-Underflow,Vulnerable,461671dfa9294ffe5bf9a4da92fbbc9173e45081adb956...
2,C000003,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000003.sol,Overflow-Underflow,Vulnerable,8043423d2d449063fbf182c76c44c79f2349524c34af47...
3,C000004,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000004.sol,Overflow-Underflow,Vulnerable,f5050f16b3daf9f3a061706e7dbb0eeff0da5bb22da51e...
4,C000005,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000005.sol,Overflow-Underflow,Vulnerable,e9516787a0af1d5746c369092a48e7b92edc21f63762d0...


In [ ]:
print("Số contract_id duy nhất:", df["contract_id"].nunique())
print("Số contract_id trùng:", df["contract_id"].duplicated().sum())

missing_files = []

for relative_path in df["copied_path"]:
    file_path = DATASET_DIR / relative_path

    if not file_path.is_file():
        missing_files.append(str(file_path))

print("Số file bị thiếu:", len(missing_files))

if missing_files:
    print("Một số file bị thiếu:")
    print(missing_files[:10])

Số contract_id duy nhất: 43232
Số contract_id trùng: 0
Số file bị thiếu: 0


**Thống kê các khai báo pragma solidity**

In [ ]:
import re
from collections import Counter

pragma_counter = Counter()
no_pragma = []
multiple_pragmas = []

total = len(df)
batch_size = 5_000

for i, row in enumerate(df.itertuples(index=False), start=1):
    file_path = DATASET_DIR / row.copied_path

    try:
        source_code = file_path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        pragmas = re.findall(
            r"\bpragma\s+solidity\s+([^;]+);",
            source_code
        )

        if not pragmas:
            no_pragma.append(row.contract_id)
        else:
            for pragma in pragmas:
                pragma_counter[pragma.strip()] += 1

            if len(pragmas) > 1:
                multiple_pragmas.append(row.contract_id)

    except OSError as e:
        print(f"Lỗi đọc {row.contract_id}: {e}")

    if i % batch_size == 0 or i == total:
        print(f"Đã xử lý {i:,}/{total:,} file ({i / total:.1%})")

print("\nHoàn tất!")
print("Số khai báo pragma khác nhau:", len(pragma_counter))
print("Số file không có pragma:", len(no_pragma))
print("Số file có nhiều pragma:", len(multiple_pragmas))

print("\n20 khai báo pragma phổ biến nhất:")
for pragma, count in pragma_counter.most_common(20):
    print(f"{count:6,} | pragma solidity {pragma};")

Đã xử lý 5,000/43,232 file (11.6%)
Đã xử lý 10,000/43,232 file (23.1%)
Đã xử lý 15,000/43,232 file (34.7%)
Đã xử lý 20,000/43,232 file (46.3%)
Đã xử lý 25,000/43,232 file (57.8%)
Đã xử lý 30,000/43,232 file (69.4%)
Đã xử lý 35,000/43,232 file (81.0%)
Đã xử lý 40,000/43,232 file (92.5%)
Đã xử lý 43,232/43,232 file (100.0%)

Hoàn tất!
Số khai báo pragma khác nhau: 211
Số file không có pragma: 1190
Số file có nhiều pragma: 1588

20 khai báo pragma phổ biến nhất:
 7,773 | pragma solidity ^0.4.24;
 6,936 | pragma solidity ^0.4.18;
 2,955 | pragma solidity ^0.4.16;
 2,835 | pragma solidity ^0.4.25;
 2,425 | pragma solidity ^0.5.0;
 2,387 | pragma solidity ^0.4.23;
 2,377 | pragma solidity ^0.4.21;
 2,226 | pragma solidity ^0.4.11;
 1,990 | pragma solidity ^0.4.19;
 1,803 | pragma solidity ^0.4.4;
 1,581 | pragma solidity ^0.4.13;
 1,441 | pragma solidity 0.4.24;
 1,130 | pragma solidity ^0.4.20;
 1,054 | pragma solidity ^0.5.2;
   989 | pragma solidity ^0.4.8;
   838 | pragma solidity ^0.4.1

In [ ]:
version_family_counter = Counter()
no_pragma_count = 0
unclassified_count = 0

total = len(df)
batch_size = 5_000

for i, row in enumerate(df.itertuples(index=False), start=1):
    file_path = DATASET_DIR / row.copied_path

    try:
        source_code = file_path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        pragmas = re.findall(
            r"\bpragma\s+solidity\s+([^;]+);",
            source_code
        )

        if not pragmas:
            no_pragma_count += 1
        else:
            versions_found = set()

            for pragma in pragmas:
                versions = re.findall(
                    r"(?<![\w.])(\d+)\.(\d+)(?:\.\d+)?",
                    pragma
                )

                for major, minor in versions:
                    versions_found.add(f"{major}.{minor}.x")

            if versions_found:
                for version_family in versions_found:
                    version_family_counter[version_family] += 1
            else:
                unclassified_count += 1

    except OSError as e:
        print(f"Lỗi đọc {row.contract_id}: {e}")

    if i % batch_size == 0 or i == total:
        print(f"Đã xử lý {i:,}/{total:,} file ({i / total:.1%})")

print("\nHoàn tất thống kê!")
print("Số file không có pragma:", no_pragma_count)
print("Số file có pragma nhưng không phân loại được:", unclassified_count)

print("\nPhân bố các nhánh phiên bản được tìm thấy:")
for version, count in version_family_counter.most_common():
    print(f"{version:10} | {count:,}")

Đã xử lý 5,000/43,232 file (11.6%)
Đã xử lý 10,000/43,232 file (23.1%)
Đã xử lý 15,000/43,232 file (34.7%)
Đã xử lý 20,000/43,232 file (46.3%)
Đã xử lý 25,000/43,232 file (57.8%)
Đã xử lý 30,000/43,232 file (69.4%)
Đã xử lý 35,000/43,232 file (81.0%)
Đã xử lý 40,000/43,232 file (92.5%)
Đã xử lý 43,232/43,232 file (100.0%)

Hoàn tất thống kê!
Số file không có pragma: 1190
Số file có pragma nhưng không phân loại được: 0

Phân bố các nhánh phiên bản được tìm thấy:
0.4.x      | 39,115
0.5.x      | 2,988
0.6.x      | 707
0.7.x      | 8
6.0.x      | 1


**Cài đặt thư viện hỗ trợ biên dịch và xử lý dũ liệu Solidity**

In [ ]:
%pip install py-solc-x pandas tqdm

In [ ]:
import solcx

print("Các phiên bản Solidity Compiler đã cài:")
print(solcx.get_installed_solc_versions())

print("\nCác phiên bản Solidity Compiler có thể cài:")
available_versions = solcx.get_installable_solc_versions()
print("Tổng số phiên bản:", len(available_versions))
print("Một số phiên bản mới nhất:", available_versions[:10])

Các phiên bản Solidity Compiler đã cài:
[<Version('0.5.17')>]

Các phiên bản Solidity Compiler có thể cài:
Tổng số phiên bản: 92
Một số phiên bản mới nhất: [<Version('0.8.37')>, <Version('0.8.36')>, <Version('0.8.35')>, <Version('0.8.34')>, <Version('0.8.33')>, <Version('0.8.32')>, <Version('0.8.31')>, <Version('0.8.30')>, <Version('0.8.29')>, <Version('0.8.28')>]


In [ ]:
versions_to_install = [
    "0.4.11",
    "0.4.24",
    "0.4.26",
    "0.5.17",
    "0.6.12",
    "0.7.6",
    "0.8.37",
]

installed_versions = {
    str(version) for version in solcx.get_installed_solc_versions()
}

for version in versions_to_install:
    if version in installed_versions:
        print(f"Đã có solc {version}, bỏ qua.")
        continue

    try:
        print(f"\nĐang cài solc {version}...")
        solcx.install_solc(version)
        print(f"Đã cài thành công solc {version}.")
    except Exception as e:
        print(f"Lỗi cài solc {version}: {e}")

print("\nCác phiên bản đã cài:")
print(solcx.get_installed_solc_versions())


Đang cài solc 0.4.11...
Đã cài thành công solc 0.4.11.

Đang cài solc 0.4.24...
Đã cài thành công solc 0.4.24.

Đang cài solc 0.4.26...
Đã cài thành công solc 0.4.26.
Đã có solc 0.5.17, bỏ qua.

Đang cài solc 0.6.12...
Đã cài thành công solc 0.6.12.

Đang cài solc 0.7.6...
Đã cài thành công solc 0.7.6.

Đang cài solc 0.8.37...
Đã cài thành công solc 0.8.37.

Các phiên bản đã cài:
[<Version('0.8.37')>, <Version('0.7.6')>, <Version('0.6.12')>, <Version('0.5.17')>, <Version('0.4.26')>, <Version('0.4.24')>, <Version('0.4.11')>]


In [ ]:
DATASET_DIR = Path(
    "/content/drive/MyDrive/IS6002/merged_dataset"
)
CONTRACTS_DIR = DATASET_DIR / "contracts"
LABELS_PATH = DATASET_DIR / "labels.csv"

df = pd.read_csv(LABELS_PATH)

print("Số contract trong labels.csv:", len(df))
print("Số file Solidity:", len(list(CONTRACTS_DIR.glob("*.sol"))))

print("\nCác phiên bản solc đã cài:")
print(solcx.get_installed_solc_versions())

print("\nPhân bố nhãn:")
print(df["label"].value_counts())

Số contract trong labels.csv: 43232
Số file Solidity: 43232

Các phiên bản solc đã cài:
[<Version('0.8.37')>, <Version('0.7.6')>, <Version('0.6.12')>, <Version('0.5.17')>, <Version('0.4.26')>, <Version('0.4.24')>, <Version('0.4.11')>]

Phân bố nhãn:
label
Vulnerable        29861
Non-vulnerable    13371
Name: count, dtype: int64


In [ ]:
%pip install evmdasm

In [ ]:
import json
import re
import time
import logging
from evmdasm import EvmBytecode
from tqdm.auto import tqdm

DATASET_DIR = Path(
    "/content/drive/MyDrive/IS6002/merged_dataset"
)
CONTRACTS_DIR = DATASET_DIR / "contracts"
LABELS_PATH = DATASET_DIR / "labels.csv"

OUTPUT_DIR = DATASET_DIR / "opcode_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(LABELS_PATH)

INSTALLED_SOLC = sorted(
    [str(v) for v in solcx.get_installed_solc_versions()]
)

print("Dataset:", DATASET_DIR.exists())
print("Số dòng nhãn:", len(labels_df))
print("Số file Solidity:", len(list(CONTRACTS_DIR.glob("*.sol"))))
print("Số contract_id duy nhất:", labels_df["contract_id"].nunique())
print("Solc đã cài:", INSTALLED_SOLC)
print("Thư mục kết quả:", OUTPUT_DIR)
print("Thư viện evmdasm đã sẵn sàng.")

Dataset: True
Số dòng nhãn: 43232
Số file Solidity: 43232
Số contract_id duy nhất: 43232
Solc đã cài: ['0.4.11', '0.4.24', '0.4.26', '0.5.17', '0.6.12', '0.7.6', '0.8.37']
Thư mục kết quả: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features
Thư viện evmdasm đã sẵn sàng.


In [ ]:
from collections import Counter
from tqdm.auto import tqdm

pragma_pattern = re.compile(
    r"\bpragma\s+solidity\s+([^;]+);",
    flags=re.IGNORECASE
)

pragma_rows = []

for file_path in tqdm(
    sorted(CONTRACTS_DIR.glob("*.sol")),
    desc="Đọc pragma"
):
    source = file_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    pragmas = pragma_pattern.findall(source)

    pragma_rows.append({
        "contract_id": file_path.stem,
        "pragma_count": len(pragmas),
        "pragma": " | ".join(p.strip() for p in pragmas)
    })

pragma_df = pd.DataFrame(pragma_rows)

print("Tổng số file:", len(pragma_df))
print("Không có pragma:", (pragma_df["pragma_count"] == 0).sum())
print("Có nhiều pragma:", (pragma_df["pragma_count"] > 1).sum())
print("Số chuỗi pragma khác nhau:", pragma_df["pragma"].nunique())

print("\n20 chuỗi pragma phổ biến nhất:")
display(
    pragma_df["pragma"]
    .value_counts()
    .head(20)
    .rename_axis("pragma")
    .reset_index(name="file_count")
)

Đọc pragma:   0%|          | 0/43232 [00:00<?, ?it/s]

Tổng số file: 43232
Không có pragma: 1190
Có nhiều pragma: 1588
Số chuỗi pragma khác nhau: 862

20 chuỗi pragma phổ biến nhất:


,pragma,file_count
0,^0.4.18,6286
1,^0.4.24,6088
2,^0.4.16,2853
3,^0.4.25,2755
4,^0.4.21,2066
5,^0.4.23,1836
6,^0.4.11,1802
7,^0.4.19,1778
8,^0.4.4,1768
9,^0.4.13,1405


In [ ]:
required_vars = [
    "CONTRACTS_DIR",
    "OUTPUT_DIR",
    "labels_df",
    "INSTALLED_SOLC",
    "select_compiler_for_source",
    "pragma_to_specifiers",
]

for name in required_vars:
    print(f"{name}: {'OK' if name in globals() else 'MISSING'}")

print("\nSố dòng nhãn:", len(labels_df) if "labels_df" in globals() else "N/A")
print("Số compiler đã cài:", len(INSTALLED_SOLC) if "INSTALLED_SOLC" in globals() else "N/A")

CONTRACTS_DIR: OK
OUTPUT_DIR: OK
labels_df: OK
INSTALLED_SOLC: OK
select_compiler_for_source: OK
pragma_to_specifiers: OK

Số dòng nhãn: 43232
Số compiler đã cài: 7


In [ ]:

BATCH_SIZE = 100

completed_ids_v2 = set()

for status_file in STATUS_BATCH_DIR.glob("status_*.csv"):
    try:
        df_status = pd.read_csv(status_file)
        if {"contract_id", "status"}.issubset(df_status.columns):
            completed_ids_v2.update(
                df_status.loc[
                    df_status["status"].isin(
                        ["FEATURE_SUCCESS", "COMPILE_NO_RUNTIME"]
                    ),
                    "contract_id"
                ].astype(str)
            )
    except Exception as exc:
        print("Không đọc được checkpoint:", status_file.name, exc)

all_ids_v2 = labels_df["contract_id"].astype(str).tolist()
pending_ids_v2 = [
    cid for cid in all_ids_v2
    if cid not in completed_ids_v2
]

print("Tổng dataset:", len(all_ids_v2))
print("Đã hoàn tất ở v2:", len(completed_ids_v2))
print("Còn lại:", len(pending_ids_v2))

start_time_v2 = time.time()
total_batches_v2 = (
    len(pending_ids_v2) + BATCH_SIZE - 1
) // BATCH_SIZE

for batch_number, start in enumerate(
    range(0, len(pending_ids_v2), BATCH_SIZE),
    start=1
):
    batch_ids = pending_ids_v2[start:start + BATCH_SIZE]
    feature_rows = []
    status_rows = []

    for cid in tqdm(
        batch_ids,
        desc=f"v2 Batch {batch_number}/{total_batches_v2}"
    ):
        feature, status = process_contract(cid)
        status_rows.append(status)
        if feature is not None:
            feature_rows.append(feature)

    feature_path = FEATURE_BATCH_DIR / f"features_{batch_number:05d}.csv"
    status_path = STATUS_BATCH_DIR / f"status_{batch_number:05d}.csv"

    if feature_rows:
        tmp_feature_path = feature_path.with_suffix(".tmp")
        pd.DataFrame(feature_rows).to_csv(tmp_feature_path, index=False)
        tmp_feature_path.replace(feature_path)

    tmp_status_path = status_path.with_suffix(".tmp")
    pd.DataFrame(status_rows).to_csv(tmp_status_path, index=False)
    tmp_status_path.replace(status_path)

    print(
        f"Batch {batch_number}/{total_batches_v2}: "
        f"{len(feature_rows)} feature thành công / "
        f"{len(status_rows)} file; "
        f"đã chạy {(time.time() - start_time_v2) / 60:.1f} phút"
    )

print("Lượt xử lý v2 kết thúc.")


Tổng dataset: 43232
Đã hoàn tất ở v2: 0
Còn lại: 43232


v2 Batch 1/433:   0%|          | 0/100 [00:00<?, ?it/s]

Batch 1/433: 100 feature thành công / 100 file; đã chạy 0.0 phút


v2 Batch 2/433:   0%|          | 0/100 [00:00<?, ?it/s]

Batch 2/433: 100 feature thành công / 100 file; đã chạy 0.1 phút


v2 Batch 3/433:   0%|          | 0/100 [00:00<?, ?it/s]

... (857 outputs left out)

v2 Batch 432/433:   0%|          | 0/100 [00:00<?, ?it/s]

Batch 432/433: 93 feature thành công / 100 file; đã chạy 24.1 phút


v2 Batch 433/433:   0%|          | 0/32 [00:00<?, ?it/s]

Batch 433/433: 31 feature thành công / 32 file; đã chạy 24.2 phút
Lượt xử lý v2 kết thúc.


In [ ]:
feature_row_counts = []
feature_ids = []

for f in sorted(FEATURE_BATCH_DIR.glob("features_*.csv")):
    df = pd.read_csv(f)
    feature_row_counts.append({
        "file": f.name,
        "rows": len(df),
        "unique_ids": df["contract_id"].nunique()
    })
    feature_ids.extend(df["contract_id"].dropna().astype(str).tolist())

feature_check = pd.DataFrame(feature_row_counts)

status_row_counts = []
status_ids = []

for f in sorted(STATUS_BATCH_DIR.glob("status_*.csv")):
    df = pd.read_csv(f)
    status_row_counts.append({
        "file": f.name,
        "rows": len(df),
        "unique_ids": df["contract_id"].nunique()
    })
    status_ids.extend(df["contract_id"].dropna().astype(str).tolist())

status_check = pd.DataFrame(status_row_counts)

print("=== FEATURE BATCH V2 ===")
print("Tổng số dòng:", feature_check["rows"].sum())
print("Tổng ID duy nhất:", len(set(feature_ids)))
print("Số ID trùng giữa các batch:", len(feature_ids) - len(set(feature_ids)))
print("Batch có số dòng khác 100 (trừ batch cuối):")
display(feature_check[feature_check["rows"] != 100])

print("\n=== STATUS BATCH V2 ===")
print("Tổng số dòng:", status_check["rows"].sum())
print("Tổng ID duy nhất:", len(set(status_ids)))
print("Số ID trùng giữa các batch:", len(status_ids) - len(set(status_ids)))
print("Tổng dòng status theo từng trạng thái:")
print(status_check["rows"].sum())


=== FEATURE BATCH V2 ===
Tổng số dòng: 39498
Tổng ID duy nhất: 39498
Số ID trùng giữa các batch: 0
Batch có số dòng khác 100 (trừ batch cuối):


,file,rows,unique_ids
3,features_00004.csv,98,98
4,features_00005.csv,95,95
5,features_00006.csv,87,87
6,features_00007.csv,87,87
7,features_00008.csv,89,89
...,...,...,...
428,features_00429.csv,87,87
429,features_00430.csv,89,89
430,features_00431.csv,93,93
431,features_00432.csv,93,93



=== STATUS BATCH V2 ===
Tổng số dòng: 43232
Tổng ID duy nhất: 43232
Số ID trùng giữa các batch: 0
Tổng dòng status theo từng trạng thái:
43232


In [ ]:
status_files = sorted(STATUS_BATCH_DIR.glob("status_*.csv"))
status_df = pd.concat(
    [pd.read_csv(f) for f in status_files],
    ignore_index=True
)

failed_df = status_df[status_df["status"] != "FEATURE_SUCCESS"].copy()

print("=== THỐNG KÊ LỖI ===")
print("Tổng contract thất bại:", len(failed_df))
print("\nSố lượng theo trạng thái:")
print(failed_df["status"].value_counts())

print("\n=== COMPILER ĐƯỢC GHI NHẬN ===")
print(failed_df["compiler"].value_counts(dropna=False).head(20))

print("\n=== MẪU THÔNG BÁO LỖI ===")
for status, group in failed_df.groupby("status", dropna=False):
    print(f"\n--- {status} ---")
    for error in group["error"].dropna().unique()[:5]:
        print(str(error)[:500])


=== THỐNG KÊ LỖI ===
Tổng contract thất bại: 3734

Số lượng theo trạng thái:
status
NO_COMPILER_CANDIDATE    3002
PROCESS_FAILED            727
COMPILE_NO_RUNTIME          5
Name: count, dtype: int64

=== COMPILER ĐƯỢC GHI NHẬN ===
compiler
NaN       3729
0.4.11       5
Name: count, dtype: int64

=== MẪU THÔNG BÁO LỖI ===

--- COMPILE_NO_RUNTIME ---

--- NO_COMPILER_CANDIDATE ---
Không có compiler candidate

--- PROCESS_FAILED ---
solc 0.4.24: Runtime bytecode không hợp lệ || solc 0.4.26: Runtime bytecode không hợp lệ
solc 0.4.11: An error occurred during execution
> command: `/root/.solcx/solc-v0.4.11 --combined-json bin-runtime --allow-paths /content/drive/MyDrive/IS6002/merged_dataset/contracts -`
> return code: `1`
> stdout:

> stderr:
<stdin>:126:65: Error: Expected token LBrace got 'View'
    function getSmartContractBy || solc 0.4.24: An error occurred during execution
> command: `/root/.solcx/solc-v0.4.24 --combined-json bin-runtime --allow-paths /content/drive/MyDrive/IS6002/m

In [ ]:
no_compiler_ids = (
    status_df.loc[
        status_df["status"] == "NO_COMPILER_CANDIDATE",
        "contract_id"
    ]
    .dropna()
    .astype(str)
    .head(20)
    .tolist()
)

print("=== KIỂM TRA PRAGMA ===")

for contract_id in no_compiler_ids:
    contract_path = CONTRACTS_DIR / f"{contract_id}.sol"

    print(f"\n--- {contract_id} ---")

    if not contract_path.exists():
        print("Không tìm thấy file")
        continue

    source = contract_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    pragmas = re.findall(
        r"\bpragma\s+solidity\s+([^;]+);",
        source
    )

    if pragmas:
        for pragma in pragmas:
            print("pragma solidity", pragma + ";")
    else:
        print("Không tìm thấy pragma solidity")

    print("Đầu file:")
    print("\n".join(source.splitlines()[:8]))


=== KIỂM TRA PRAGMA ===

--- C000359 ---
pragma solidity 0.4.9;
Đầu file:
/*
 * @source: https://github.com/paritytech/parity-ethereum/blob/4d08e7b0aec46443bf26547b17d10cb302672835/js/src/contracts/snippets/enhanced-wallet.sol#L216
 * @author: parity
 * @vulnerable_at_lines: 223,437
 */

//sol Wallet
// Multi-sig, daily-limited account proxy/wallet.

--- C000379 ---
pragma solidity 0.4.25;
Đầu file:
/*
 * @source: https://smartcontractsecurity.github.io/SWC-registry/docs/SWC-101#overflow-simple-addsol
 * @author: -
 * @vulnerable_at_lines: 14
 */

pragma solidity 0.4.25;


--- C000493 ---
pragma solidity 0.4.25;
Đầu file:
/*
 * @source: https://smartcontractsecurity.github.io/SWC-registry/docs/SWC-104#unchecked-return-valuesol
 * @author: -
 * @vulnerable_at_lines: 17
 */

pragma solidity 0.4.25;


--- C000497 ---
pragma solidity 0.5.0;
pragma solidity 0.5.0;
pragma solidity 0.5.0;
pragma solidity 0.5.0;
pragma solidity 0.5.0;
pragma solidity 0.5.0;
pragma solidity 0.5.0;
pragma solidi

In [ ]:
from solcx import get_installed_solc_versions

installed = sorted(
    str(v) for v in get_installed_solc_versions()
)

print("=== COMPILER ĐÃ CÀI ===")
print(installed)

print("\n=== PHIÊN BẢN CÒN THIẾU TRONG CÁC MẪU ===")
sample_versions = [
    "0.4.9", "0.4.14", "0.4.17", "0.4.19",
    "0.4.21", "0.4.25", "0.5.0", "0.5.2",
    "0.5.6", "0.5.7", "0.5.8"
]

for version in sample_versions:
    print(
        version,
        "ĐÃ CÀI" if version in installed else "CHƯA CÀI"
    )


=== COMPILER ĐÃ CÀI ===
['0.4.11', '0.4.24', '0.4.26', '0.5.17', '0.6.12', '0.7.6', '0.8.37']

=== PHIÊN BẢN CÒN THIẾU TRONG CÁC MẪU ===
0.4.9 CHƯA CÀI
0.4.14 CHƯA CÀI
0.4.17 CHƯA CÀI
0.4.19 CHƯA CÀI
0.4.21 CHƯA CÀI
0.4.25 CHƯA CÀI
0.5.0 CHƯA CÀI
0.5.2 CHƯA CÀI
0.5.6 CHƯA CÀI
0.5.7 CHƯA CÀI
0.5.8 CHƯA CÀI


In [ ]:

from solcx import (
    get_installed_solc_versions,
    install_solc
)

versions_to_install = [
    "0.4.9",
    "0.4.14",
    "0.4.17",
    "0.4.19",
    "0.4.21",
    "0.4.25",
    "0.5.0",
    "0.5.2",
    "0.5.6",
    "0.5.7",
    "0.5.8",
]

installed = {
    str(v) for v in get_installed_solc_versions()
}

for version in versions_to_install:
    if version in installed:
        print(f"Đã có solc {version}, bỏ qua.")
        continue

    print(f"Đang cài solc {version}...")
    try:
        install_solc(version)
        print(f"Cài thành công solc {version}")
    except Exception as e:
        print(f"Cài thất bại solc {version}: {e}")

print("\n=== DANH SÁCH SAU KHI CÀI ===")
print(sorted(str(v) for v in get_installed_solc_versions()))


Đang cài solc 0.4.9...
Cài thất bại solc 0.4.9: py-solc-x does not support solc versions <0.4.11
Đang cài solc 0.4.14...
Cài thành công solc 0.4.14
Đang cài solc 0.4.17...
Cài thành công solc 0.4.17
Đang cài solc 0.4.19...
Cài thành công solc 0.4.19
Đang cài solc 0.4.21...
Cài thành công solc 0.4.21
Đang cài solc 0.4.25...
Cài thành công solc 0.4.25
Đang cài solc 0.5.0...
Cài thành công solc 0.5.0
Đang cài solc 0.5.2...
Cài thành công solc 0.5.2
Đang cài solc 0.5.6...
Cài thành công solc 0.5.6
Đang cài solc 0.5.7...
Cài thành công solc 0.5.7
Đang cài solc 0.5.8...
Cài thành công solc 0.5.8

=== DANH SÁCH SAU KHI CÀI ===
['0.4.11', '0.4.14', '0.4.17', '0.4.19', '0.4.21', '0.4.24', '0.4.25', '0.4.26', '0.5.0', '0.5.17', '0.5.2', '0.5.6', '0.5.7', '0.5.8', '0.6.12', '0.7.6', '0.8.37']


In [ ]:
import inspect

print("=== pragma_to_specifiers ===")
print(inspect.getsource(pragma_to_specifiers))

print("\n=== extract_pragmas ===")
print(inspect.getsource(extract_pragmas))

print("\n=== select_compiler_for_source ===")
print(inspect.getsource(select_compiler_for_source))

print("\n=== get_compiler_candidates ===")
print(inspect.getsource(get_compiler_candidates))


=== pragma_to_specifiers ===
def pragma_to_specifiers(pragma_text):
    if not pragma_text:
        return None

    text = pragma_text.strip()

    if not re.fullmatch(r"[\^~><=!*0-9.\s,]+", text):
        return None

    parts = re.findall(
        r"(\^|~|>=|<=|>|<|=)?\s*(\d+(?:\.\d+){0,2})",
        text
    )

    if not parts:
        return None

    constraints = []

    try:
        for operator, version_text in parts:
            version = normalize_solidity_version(version_text)
            major, minor, patch = version.release[:3]

            if operator == "^":
                constraints.append(f">={version}")
                if major > 0:
                    constraints.append(f"<{major + 1}.0.0")
                elif minor > 0:
                    constraints.append(f"<0.{minor + 1}.0")
                else:
                    constraints.append(f"<0.0.{patch + 1}")

            elif operator == "~":
                constraints.extend([
                    f">={versi

In [ ]:
from packaging.version import Version
from packaging.specifiers import SpecifierSet
from solcx import get_installed_solc_versions

INSTALLED_SOLC = sorted(
    str(v) for v in get_installed_solc_versions()
)
compiler_versions = [Version(v) for v in INSTALLED_SOLC]

print("Compiler hiện có:", INSTALLED_SOLC)


def pragma_to_specifiers(pragma_text):
    if not pragma_text:
        return None

    text = pragma_text.strip()

    text = re.sub(r"\s+", "", text)

    if not re.fullmatch(r"[\^~><=!*0-9.,]+", text):
        return None

    parts = re.findall(
        r"(\^|~|>=|<=|>|<|=)?(\d+(?:\.\d+){0,2})",
        text
    )

    if not parts:
        return None

    reconstructed = "".join(op + ver for op, ver in parts)
    if reconstructed != text:
        return None

    constraints = []

    try:
        for operator, version_text in parts:
            version = normalize_solidity_version(version_text)
            major, minor, patch = version.release[:3]

            if operator == "^":
                constraints.append(f">={version}")
                if major > 0:
                    constraints.append(f"<{major + 1}.0.0")
                elif minor > 0:
                    constraints.append(f"<0.{minor + 1}.0")
                else:
                    constraints.append(f"<0.0.{patch + 1}")

            elif operator == "~":
                constraints.extend([
                    f">={version}",
                    f"<{major}.{minor + 1}.0"
                ])

            elif operator:
                constraints.append(f"{operator}{version}")

            else:
                constraints.append(f"=={version}")

        return SpecifierSet(",".join(constraints))

    except Exception:
        return None


test_pragmas = [
    "0.4.9",
    "0.4.25",
    "0.5.0",
    "^ 0.4 .2",
    "^ 0.5 .1",
    ">=0.4.22 <0.6.0",
    "^0.5.11",
]

print("\n=== KIỂM TRA PARSER ===")
for pragma in test_pragmas:
    spec = pragma_to_specifiers(pragma)
    matching = (
        [v for v in compiler_versions if v in spec]
        if spec is not None else []
    )
    print(f"{pragma!r} -> {spec}; compiler phù hợp: {matching}")


Compiler hiện có: ['0.4.11', '0.4.14', '0.4.17', '0.4.19', '0.4.21', '0.4.24', '0.4.25', '0.4.26', '0.5.0', '0.5.17', '0.5.2', '0.5.6', '0.5.7', '0.5.8', '0.6.12', '0.7.6', '0.8.37']

=== KIỂM TRA PARSER ===
'0.4.9' -> ==0.4.9; compiler phù hợp: []
'0.4.25' -> ==0.4.25; compiler phù hợp: [<Version('0.4.25')>]
'0.5.0' -> ==0.5.0; compiler phù hợp: [<Version('0.5.0')>]
'^ 0.4 .2' -> <0.5.0,>=0.4.2; compiler phù hợp: [<Version('0.4.11')>, <Version('0.4.14')>, <Version('0.4.17')>, <Version('0.4.19')>, <Version('0.4.21')>, <Version('0.4.24')>, <Version('0.4.25')>, <Version('0.4.26')>]
'^ 0.5 .1' -> <0.6.0,>=0.5.1; compiler phù hợp: [<Version('0.5.17')>, <Version('0.5.2')>, <Version('0.5.6')>, <Version('0.5.7')>, <Version('0.5.8')>]
'>=0.4.22 <0.6.0' -> <0.6.0,>=0.4.22; compiler phù hợp: [<Version('0.4.24')>, <Version('0.4.25')>, <Version('0.4.26')>, <Version('0.5.0')>, <Version('0.5.17')>, <Version('0.5.2')>, <Version('0.5.6')>, <Version('0.5.7')>, <Version('0.5.8')>]
'^0.5.11' -> <0.6.0,>=

In [ ]:
compiler_versions = sorted(
    [Version(v) for v in INSTALLED_SOLC]
)

no_compiler_df = status_df[
    status_df["status"] == "NO_COMPILER_CANDIDATE"
].copy()

results = []

for contract_id in no_compiler_df["contract_id"].astype(str):
    path = CONTRACTS_DIR / f"{contract_id}.sol"

    if not path.exists():
        results.append({
            "contract_id": contract_id,
            "result": "FILE_NOT_FOUND",
            "candidate_count": 0,
            "candidates": ""
        })
        continue

    source = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    candidates = get_compiler_candidates(source)

    results.append({
        "contract_id": contract_id,
        "result": (
            "HAS_CANDIDATE"
            if candidates else "NO_CANDIDATE"
        ),
        "candidate_count": len(candidates),
        "candidates": ", ".join(candidates)
    })

candidate_check_df = pd.DataFrame(results)

print("=== KẾT QUẢ KIỂM TRA CANDIDATE ===")
print(candidate_check_df["result"].value_counts(dropna=False))

print("\n=== TỶ LỆ TÌM ĐƯỢC COMPILER ===")
print(
    "Có candidate:",
    (candidate_check_df["result"] == "HAS_CANDIDATE").sum()
)
print(
    "Không có candidate:",
    (candidate_check_df["result"] == "NO_CANDIDATE").sum()
)
print(
    "Không tìm thấy file:",
    (candidate_check_df["result"] == "FILE_NOT_FOUND").sum()
)

print("\n=== 20 CONTRACT ĐẦU TIÊN ===")
display(candidate_check_df.head(20))


=== KẾT QUẢ KIỂM TRA CANDIDATE ===
result
HAS_CANDIDATE    1757
NO_CANDIDATE     1245
Name: count, dtype: int64

=== TỶ LỆ TÌM ĐƯỢC COMPILER ===
Có candidate: 1757
Không có candidate: 1245
Không tìm thấy file: 0

=== 20 CONTRACT ĐẦU TIÊN ===


,contract_id,result,candidate_count,candidates
0,C000359,NO_CANDIDATE,0,
1,C000379,HAS_CANDIDATE,1,0.4.25
2,C000493,HAS_CANDIDATE,1,0.4.25
3,C000497,HAS_CANDIDATE,1,0.5.0
4,C000498,HAS_CANDIDATE,1,0.5.6
5,C000500,HAS_CANDIDATE,1,0.5.8
6,C000509,HAS_CANDIDATE,1,0.5.6
7,C000511,HAS_CANDIDATE,1,0.4.25
8,C000512,HAS_CANDIDATE,1,0.5.8
9,C000513,HAS_CANDIDATE,1,0.4.25


In [ ]:
import inspect

print(inspect.getsource(get_compiler_candidates))
print("\n=== process_contract ===")
print(inspect.getsource(process_contract))


def get_compiler_candidates(source):
    pragmas = extract_pragmas(source)

    # Không có pragma hoặc pragma chưa được parser hỗ trợ:
    # thử các compiler đã cài, từ cũ đến mới.
    if not pragmas:
        return [str(v) for v in compiler_versions]

    specs = [pragma_to_specifiers(p) for p in pragmas]

    if any(spec is None for spec in specs):
        return [str(v) for v in compiler_versions]

    return [
        str(v) for v in compiler_versions
        if all(v in spec for spec in specs)
    ]


=== process_contract ===
def process_contract(contract_id):
    source_path = CONTRACTS_DIR / f"{contract_id}.sol"

    status = {
        "contract_id": contract_id,
        "status": None,
        "compiler": None,
        "runtime_contracts": 0,
        "metadata_removed": 0,
        "metadata_unverified": 0,
        "unknown_opcode_count": 0,
        "error": None,
    }

    if not source_path.exists():
        status["status"] = "FILE_NOT_FOUND"
        status["error"] = str(so

In [ ]:
successful_ids = set(
    feature_df["contract_id"].astype(str)
)

retry_ids = candidate_check_df.loc[
    candidate_check_df["result"] == "HAS_CANDIDATE",
    "contract_id"
].astype(str).tolist()

retry_ids = [
    cid for cid in retry_ids
    if cid not in successful_ids
]

print("Số contract có candidate ban đầu:",
      (candidate_check_df["result"] == "HAS_CANDIDATE").sum())

print("Số contract đã thành công trước đó:",
      sum(cid in successful_ids for cid in retry_ids))

print("Số contract cần retry:", len(retry_ids))
print("Một số ID:", retry_ids[:10])


Số contract có candidate ban đầu: 1757
Số contract đã thành công trước đó: 0
Số contract cần retry: 1757
Một số ID: ['C000379', 'C000493', 'C000497', 'C000498', 'C000500', 'C000509', 'C000511', 'C000512', 'C000513', 'C000551']


In [ ]:

from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

RETRY_FEATURE_DIR = (
    Path("/content/drive/MyDrive/IS6002/merged_dataset/")
    / "opcode_features/feature_batches_retry"
)
RETRY_STATUS_DIR = (
    Path("/content/drive/MyDrive/IS6002/merged_dataset/")
    / "opcode_features/status_batches_retry"
)

RETRY_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
RETRY_STATUS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 100

retry_batches = [
    retry_ids[i:i + BATCH_SIZE]
    for i in range(0, len(retry_ids), BATCH_SIZE)
]

print("Tổng contract:", len(retry_ids))
print("Tổng batch:", len(retry_batches))
print("Thư mục feature:", RETRY_FEATURE_DIR)
print("Thư mục status:", RETRY_STATUS_DIR)


Tổng contract: 1757
Tổng batch: 18
Thư mục feature: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/feature_batches_retry
Thư mục status: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/status_batches_retry


In [ ]:

import json
import time
import pandas as pd
from tqdm.auto import tqdm

start_time = time.time()
retry_summary = []

for batch_idx, batch_ids in enumerate(retry_batches, start=1):
    feature_path = RETRY_FEATURE_DIR / f"features_{batch_idx:03d}.csv"
    status_path = RETRY_STATUS_DIR / f"status_{batch_idx:03d}.csv"

    if feature_path.exists() and status_path.exists():
        print(f"Batch {batch_idx:03d}: đã có kết quả, bỏ qua")
        continue

    batch_features = []
    batch_statuses = []

    for contract_id in tqdm(
        batch_ids,
        desc=f"Retry batch {batch_idx:03d}/{len(retry_batches)}",
        leave=False
    ):
        feature, status = process_contract(contract_id)
        batch_statuses.append(status)

        if feature is not None:
            batch_features.append(feature)

    pd.DataFrame(batch_features).to_csv(
        feature_path, index=False
    )
    pd.DataFrame(batch_statuses).to_csv(
        status_path, index=False
    )

    status_counts = pd.Series(
        [s["status"] for s in batch_statuses]
    ).value_counts().to_dict()

    retry_summary.append({
        "batch": batch_idx,
        "contracts": len(batch_ids),
        "features": len(batch_features),
        "status_counts": status_counts
    })

    print(
        f"Batch {batch_idx:03d}: "
        f"{len(batch_features)}/{len(batch_ids)} feature thành công"
    )

print(f"\nThời gian chạy: {(time.time() - start_time) / 60:.1f} phút")
print("Retry hoàn tất hoặc đã xử lý hết các batch chưa lưu.")


Retry batch 001/18:   0%|          | 0/100 [00:00<?, ?it/s]

Batch 001: 99/100 feature thành công


Retry batch 002/18:   0%|          | 0/100 [00:00<?, ?it/s]

Batch 002: 96/100 feature thành công


Retry batch 003/18:   0%|          | 0/100 [00:00<?, ?it/s]

Batch 003: 99/100 feature thành công


... (26 outputs left out)

Retry batch 017/18:   0%|          | 0/100 [00:00<?, ?it/s]

Batch 017: 99/100 feature thành công


Retry batch 018/18:   0%|          | 0/57 [00:00<?, ?it/s]

Batch 018: 57/57 feature thành công

Thời gian chạy: 1.5 phút
Retry hoàn tất hoặc đã xử lý hết các batch chưa lưu.


In [ ]:
retry_feature_files = sorted(RETRY_FEATURE_DIR.glob("features_*.csv"))
retry_status_files = sorted(RETRY_STATUS_DIR.glob("status_*.csv"))

retry_feature_df = pd.concat(
    [pd.read_csv(p) for p in retry_feature_files],
    ignore_index=True
) if retry_feature_files else pd.DataFrame()

retry_status_df = pd.concat(
    [pd.read_csv(p) for p in retry_status_files],
    ignore_index=True
) if retry_status_files else pd.DataFrame()

retry_feature_df["contract_id"] = (
    retry_feature_df["contract_id"].astype(str)
)

feature_df["contract_id"] = feature_df["contract_id"].astype(str)

print("=== KIỂM TRA FILE RETRY ===")
print("Số file feature:", len(retry_feature_files))
print("Số file status:", len(retry_status_files))
print("Số feature retry:", len(retry_feature_df))
print("ID retry duy nhất:", retry_feature_df["contract_id"].nunique())
print("ID trùng trong retry:",
      retry_feature_df["contract_id"].duplicated().sum())

combined_features_df = pd.concat(
    [feature_df, retry_feature_df],
    ignore_index=True,
    sort=False
)

print("\n=== TỔNG FEATURE SAU RETRY ===")
print("Tổng số dòng:", len(combined_features_df))
print("Số contract duy nhất:",
      combined_features_df["contract_id"].nunique())
print("ID trùng sau khi gộp:",
      combined_features_df["contract_id"].duplicated().sum())

print("\n=== TRẠNG THÁI RETRY ===")
print(
    retry_status_df["status"]
    .value_counts(dropna=False)
    .to_string()
)


=== KIỂM TRA FILE RETRY ===
Số file feature: 18
Số file status: 18
Số feature retry: 1720
ID retry duy nhất: 1720
ID trùng trong retry: 0

=== TỔNG FEATURE SAU RETRY ===
Tổng số dòng: 41218
Số contract duy nhất: 41218
ID trùng sau khi gộp: 0

=== TRẠNG THÁI RETRY ===
status
FEATURE_SUCCESS    1720
PROCESS_FAILED       37


In [ ]:
status_df["contract_id"] = status_df["contract_id"].astype(str)
retry_status_df["contract_id"] = retry_status_df["contract_id"].astype(str)

retry_ids_set = set(retry_status_df["contract_id"])

base_status_remaining = status_df[
    ~status_df["contract_id"].isin(retry_ids_set)
].copy()

final_status_df = pd.concat(
    [base_status_remaining, retry_status_df],
    ignore_index=True
)

print("=== TRẠNG THÁI CUỐI CÙNG ===")
print(final_status_df["status"].value_counts(dropna=False).to_string())

print("\nTổng dòng trạng thái:", len(final_status_df))
print("Số contract duy nhất:", final_status_df["contract_id"].nunique())
print("ID trùng:", final_status_df["contract_id"].duplicated().sum())

print("\nSố contract trong dataset gốc:", len(labels_df))
print(
    "Contract chưa có trạng thái:",
    len(set(labels_df["contract_id"].astype(str))
        - set(final_status_df["contract_id"]))
)


=== TRẠNG THÁI CUỐI CÙNG ===
status
FEATURE_SUCCESS          41218
NO_COMPILER_CANDIDATE     1245
PROCESS_FAILED             764
COMPILE_NO_RUNTIME           5

Tổng dòng trạng thái: 43232
Số contract duy nhất: 43232
ID trùng: 0

Số contract trong dataset gốc: 43232
Contract chưa có trạng thái: 0


In [ ]:

no_compiler_ids = final_status_df.loc[
    final_status_df["status"] == "NO_COMPILER_CANDIDATE",
    "contract_id"
].astype(str).tolist()


pragma_counter = Counter()
no_pragma_count = 0
unreadable_count = 0

for contract_id in no_compiler_ids:
    path = CONTRACTS_DIR / f"{contract_id}.sol"

    try:
        source = path.read_text(
            encoding="utf-8",
            errors="replace"
        )
        pragmas = extract_pragmas(source)

        if pragmas:
            for pragma in pragmas:
                pragma_counter[pragma.strip()] += 1
        else:
            no_pragma_count += 1

    except Exception:
        unreadable_count += 1

print("=== THỐNG KÊ PRAGMA ===")
print("Contract không có pragma:", no_pragma_count)
print("Contract không đọc được:", unreadable_count)
print("Số dạng pragma khác nhau:", len(pragma_counter))

print("\n=== 30 PRAGMA PHỔ BIẾN NHẤT ===")
for pragma, count in pragma_counter.most_common(30):
    print(f"{count:4} | {pragma}")

print("\n=== COMPILER ĐANG CÀI ===")
print(INSTALLED_SOLC)


=== THỐNG KÊ PRAGMA ===
Contract không có pragma: 0
Contract không đọc được: 0
Số dạng pragma khác nhau: 31

=== 30 PRAGMA PHỔ BIẾN NHẤT ===
 480 | 0.4.18
 308 | 0.4.23
 167 | 0.4.15
 148 | 0.4.20
 124 | 0.5.4
  63 | ^0.5.2
  57 | 0.4.16
  54 | 0.5.9
  41 | ^0.5.0
  39 | 0.5.1
  27 | 0.4.8
  26 | 0.5.3
  25 | ^0.4.18
  24 | ^0.4.15
  18 | 0.5.5
  12 | 0.4.4
  12 | ^0.4.21
  11 | 0.4.22
  10 | 0.4.10
   7 | 0.4.13
   6 | ^0.4.0
   6 | 0.5.10
   6 | ^0.4.24
   5 | 0.4.12
   4 | >=0.4.24
   2 | 0.4.9
   2 | 0.4.6
   2 | ^0.5.5
   1 | ^0.5.4
   1 | >= 0.4.24

=== COMPILER ĐANG CÀI ===
['0.4.11', '0.4.14', '0.4.17', '0.4.19', '0.4.21', '0.4.24', '0.4.25', '0.4.26', '0.5.0', '0.5.17', '0.5.2', '0.5.6', '0.5.7', '0.5.8', '0.6.12', '0.7.6', '0.8.37']


In [ ]:

# Cập nhật danh sách compiler sau khi cài thêm
INSTALLED_SOLC = sorted(
    str(v) for v in solcx.get_installed_solc_versions()
)

compiler_versions = sorted(
    [Version(v) for v in INSTALLED_SOLC]
)

# Lấy các contract hiện chưa có compiler candidate
no_compiler_ids = final_status_df.loc[
    final_status_df["status"] == "NO_COMPILER_CANDIDATE",
    "contract_id"
].astype(str).tolist()

results_after_install = []

for contract_id in tqdm(no_compiler_ids, desc="Kiểm tra candidate mới"):
    path = CONTRACTS_DIR / f"{contract_id}.sol"

    source = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    candidates = get_compiler_candidates(source)

    results_after_install.append({
        "contract_id": contract_id,
        "result": (
            "HAS_CANDIDATE" if candidates else "NO_CANDIDATE"
        ),
        "candidate_count": len(candidates),
        "candidates": ", ".join(candidates)
    })

candidate_check_v2_df = pd.DataFrame(results_after_install)

print("=== KẾT QUẢ SAU KHI CÀI COMPILER ===")
print(candidate_check_v2_df["result"].value_counts())

print("\nCó candidate:",
      (candidate_check_v2_df["result"] == "HAS_CANDIDATE").sum())
print("Chưa có candidate:",
      (candidate_check_v2_df["result"] == "NO_CANDIDATE").sum())

print("\n=== CÁC CONTRACT VẪN CHƯA CÓ CANDIDATE ===")
display(
    candidate_check_v2_df[
        candidate_check_v2_df["result"] == "NO_CANDIDATE"
    ].head(20)
)


Kiểm tra candidate mới:   0%|          | 0/1245 [00:00<?, ?it/s]

=== KẾT QUẢ SAU KHI CÀI COMPILER ===
result
HAS_CANDIDATE    1188
NO_CANDIDATE       57
Name: count, dtype: int64

Có candidate: 1188
Chưa có candidate: 57

=== CÁC CONTRACT VẪN CHƯA CÓ CANDIDATE ===


,contract_id,result,candidate_count,candidates
0,C000359,NO_CANDIDATE,0,
8,C000823,NO_CANDIDATE,0,
57,C002244,NO_CANDIDATE,0,
61,C002346,NO_CANDIDATE,0,
116,C004021,NO_CANDIDATE,0,
150,C005356,NO_CANDIDATE,0,
152,C005508,NO_CANDIDATE,0,
172,C006113,NO_CANDIDATE,0,
177,C006248,NO_CANDIDATE,0,
211,C007219,NO_CANDIDATE,0,


In [ ]:

remaining_ids = candidate_check_v2_df.loc[
    candidate_check_v2_df["result"] == "NO_CANDIDATE",
    "contract_id"
].tolist()

remaining_pragma_rows = []

for contract_id in remaining_ids:
    path = CONTRACTS_DIR / f"{contract_id}.sol"
    source = path.read_text(encoding="utf-8", errors="replace")

    remaining_pragma_rows.append({
        "contract_id": contract_id,
        "pragmas": " | ".join(extract_pragmas(source)) or "(không tìm thấy pragma)",
        "source_length": len(source)
    })

remaining_pragma_df = pd.DataFrame(remaining_pragma_rows)

print("Số contract:", len(remaining_pragma_df))
print("\n=== Thống kê pragma ===")
display(
    remaining_pragma_df["pragmas"]
    .value_counts()
    .rename_axis("pragma")
    .reset_index(name="count")
)

print("\n=== Chi tiết 57 contract ===")
display(remaining_pragma_df)


Số contract: 57

=== Thống kê pragma ===


,pragma,count
0,0.4.8,27
1,0.4.4,12
2,0.4.10,10
3,^0.5.2 | ^0.4.24,3
4,0.4.9,2
5,0.4.6,2
6,^0.4.24 | ^0.5.0 | ^0.4.24 | ^0.5.0 | >=0.4.24...,1



=== Chi tiết 57 contract ===


,contract_id,pragmas,source_length
0,C000359,0.4.9,16509
1,C000823,0.4.10,18211
2,C002244,0.4.8,3286
3,C002346,0.4.8,19814
4,C004021,0.4.4,5655
5,C005356,0.4.10,8204
6,C005508,0.4.8,2984
7,C006113,0.4.4,7190
8,C006248,0.4.8,20154
9,C007219,0.4.8,19805


In [ ]:

import requests

url = "https://binaries.soliditylang.org/linux-amd64/list.json"

try:
    response = requests.get(url, timeout=30)
    print("HTTP status:", response.status_code)
    response.raise_for_status()

    compiler_list = response.json()
    releases = compiler_list.get("releases", {})

    for version in ["0.4.4", "0.4.6", "0.4.8", "0.4.9", "0.4.10"]:
        filename = releases.get(version)
        print(f"solc {version}: {filename or 'Không tìm thấy trong danh sách'}")

except Exception as exc:
    print("Không truy cập được danh sách compiler:", repr(exc))


HTTP status: 200
solc 0.4.4: Không tìm thấy trong danh sách
solc 0.4.6: Không tìm thấy trong danh sách
solc 0.4.8: Không tìm thấy trong danh sách
solc 0.4.9: Không tìm thấy trong danh sách
solc 0.4.10: solc-linux-amd64-v0.4.10+commit.9e8cc01b


In [ ]:
import hashlib
import os
import stat
import subprocess

version = "0.4.10"
filename = releases = None

list_url = "https://binaries.soliditylang.org/linux-amd64/list.json"
data = requests.get(list_url, timeout=30)
data.raise_for_status()
releases = data.json()["releases"]

filename = releases.get(version)
if not filename:
    raise RuntimeError(f"Không tìm thấy binary solc {version}")

binary_url = f"https://binaries.soliditylang.org/linux-amd64/{filename}"
binary_path = f"/content/solc-{version}"

print("Binary:", filename)
print("URL:", binary_url)

download = requests.get(binary_url, timeout=60)
download.raise_for_status()

with open(binary_path, "wb") as f:
    f.write(download.content)

os.chmod(
    binary_path,
    os.stat(binary_path).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
)

result = subprocess.run(
    [binary_path, "--version"],
    capture_output=True,
    text=True,
    timeout=30
)

print("\nExit code:", result.returncode)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:1000])


Binary: solc-linux-amd64-v0.4.10+commit.9e8cc01b
URL: https://binaries.soliditylang.org/linux-amd64/solc-linux-amd64-v0.4.10+commit.9e8cc01b

Exit code: 0
solc, the solidity compiler commandline interface
Version: 0.4.10+commit.9e8cc01b.Linux.g++



In [ ]:

import subprocess
contract_id = "C000823"
source_path = CONTRACTS_DIR / f"{contract_id}.sol"
source = source_path.read_text(encoding="utf-8", errors="replace")

result = subprocess.run(
    ["/content/solc-0.4.10", "--combined-json", "bin-runtime"],
    input=source,
    capture_output=True,
    text=True,
    timeout=60
)

print("Contract:", contract_id)
print("Exit code:", result.returncode)

if result.stderr:
    print("\nCompiler messages:")
    print(result.stderr[:3000])

if result.returncode == 0:
    try:
        output = json.loads(result.stdout)
        contracts = output.get("contracts", {})
        runtime_count = sum(
            bool(item.get("bin-runtime", "").strip())
            for item in contracts.values()
        )

        print("\nSố contract được compiler trả về:", len(contracts))
        print("Số runtime bytecode không rỗng:", runtime_count)

        for name, item in list(contracts.items())[:5]:
            runtime = item.get("bin-runtime", "")
            print(
                name,
                "| runtime bytes:",
                len(runtime) // 2
            )

    except Exception as exc:
        print("Không đọc được JSON output:", repr(exc))
        print(result.stdout[:1000])
else:
    print("\nBiên dịch thất bại.")


Contract: C000823
Exit code: 0

Số contract được compiler trả về: 3
Số runtime bytecode không rỗng: 3
<stdin>:MultiSigWallet | runtime bytes: 8238
<stdin>:MultiSigWalletWithTimeLock | runtime bytes: 8751
<stdin>:MultiSigWalletWithTimeLockExceptRemoveAuthorizedAddress | runtime bytes: 10166


In [ ]:
url = "https://binaries.soliditylang.org/linux-amd64/list.json"
response = requests.get(url, timeout=30)
response.raise_for_status()

compiler_data = response.json()
builds = compiler_data.get("builds", [])

for version in ["0.4.4", "0.4.6", "0.4.8", "0.4.9", "0.4.10"]:
    matches = [
        item for item in builds
        if item.get("version") == version
    ]

    if matches:
        for item in matches:
            print(f"{version}: {item['path']}")
    else:
        print(f"{version}: Không tìm thấy trong builds")


0.4.4: Không tìm thấy trong builds
0.4.6: Không tìm thấy trong builds
0.4.8: Không tìm thấy trong builds
0.4.9: Không tìm thấy trong builds
0.4.10: solc-linux-amd64-v0.4.10+commit.9e8cc01b


In [ ]:
from tqdm.auto import tqdm

ids_0410 = remaining_pragma_df.loc[
    remaining_pragma_df["pragmas"] == "0.4.10",
    "contract_id"
].tolist()

test_results_0410 = []

for contract_id in tqdm(ids_0410, desc="Thử solc 0.4.10"):
    source_path = CONTRACTS_DIR / f"{contract_id}.sol"
    source = source_path.read_text(encoding="utf-8", errors="replace")

    try:
        result = subprocess.run(
            ["/content/solc-0.4.10", "--combined-json", "bin-runtime"],
            input=source,
            capture_output=True,
            text=True,
            timeout=60
        )

        contracts = {}
        if result.returncode == 0:
            try:
                contracts = json.loads(result.stdout).get("contracts", {})
            except json.JSONDecodeError:
                pass

        runtime_count = sum(
            bool(item.get("bin-runtime", "").strip())
            for item in contracts.values()
        )

        test_results_0410.append({
            "contract_id": contract_id,
            "success": runtime_count > 0,
            "runtime_count": runtime_count,
            "error": result.stderr[:300] if runtime_count == 0 else ""
        })

    except Exception as exc:
        test_results_0410.append({
            "contract_id": contract_id,
            "success": False,
            "runtime_count": 0,
            "error": str(exc)[:300]
        })

test_results_0410_df = pd.DataFrame(test_results_0410)

print("\n=== KẾT QUẢ ===")
print(test_results_0410_df["success"].value_counts())
display(test_results_0410_df)


Thử solc 0.4.10:   0%|          | 0/10 [00:00<?, ?it/s]


=== KẾT QUẢ ===
success
True    10
Name: count, dtype: int64


,contract_id,success,runtime_count,error
0,C000823,True,3,
1,C005356,True,1,
2,C016118,True,1,
3,C016520,True,1,
4,C017780,True,1,
5,C017816,True,2,
6,C018773,True,1,
7,C020083,True,1,
8,C022574,True,2,
9,C034032,True,1,


In [ ]:
MANUAL_FEATURE_DIR = (
    Path("/content/drive/MyDrive/IS6002/merged_dataset/opcode_features")
    / "feature_batches_manual_0410"
)
MANUAL_STATUS_DIR = (
    Path("/content/drive/MyDrive/IS6002/merged_dataset/opcode_features")
    / "status_batches_manual_0410"
)

MANUAL_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_STATUS_DIR.mkdir(parents=True, exist_ok=True)

manual_feature_rows = []
manual_status_rows = []

for contract_id in tqdm(ids_0410, desc="Trích xuất opcode features"):
    source_path = CONTRACTS_DIR / f"{contract_id}.sol"
    source = source_path.read_text(encoding="utf-8", errors="replace")

    status = {
        "contract_id": contract_id,
        "status": "PROCESS_FAILED",
        "compiler": "0.4.10",
        "runtime_contracts": 0,
        "metadata_removed": 0,
        "metadata_unverified": 0,
        "unknown_opcode_count": 0,
        "error": ""
    }

    try:
        result = subprocess.run(
            ["/content/solc-0.4.10", "--combined-json", "bin-runtime"],
            input=source,
            capture_output=True,
            text=True,
            timeout=60
        )

        if result.returncode != 0:
            raise RuntimeError(result.stderr[:1000])

        compiled = json.loads(result.stdout).get("contracts", {})
        opcode_totals = {}
        runtime_count = 0

        for artifact in compiled.values():
            runtime_hex = artifact.get("bin-runtime", "").strip()
            if not runtime_hex:
                continue

            runtime_count += 1
            runtime_hex, metadata_removed, metadata_reason = (
                strip_solidity_metadata(runtime_hex)
            )

            if metadata_removed:
                status["metadata_removed"] += 1
            elif metadata_reason:
                status["metadata_unverified"] += 1

            counts = extract_opcode_counts(runtime_hex)

            for opcode, count in counts.items():
                opcode_totals[opcode] = (
                    opcode_totals.get(opcode, 0) + count
                )

        if runtime_count == 0:
            status["status"] = "COMPILE_NO_RUNTIME"
        else:
            status["status"] = "FEATURE_SUCCESS"
            status["runtime_contracts"] = runtime_count
            status["unknown_opcode_count"] = sum(
                count for opcode, count in opcode_totals.items()
                if opcode.startswith("UNKNOWN_")
            )

            manual_feature_rows.append({
                "contract_id": contract_id,
                **opcode_totals
            })

    except Exception as exc:
        status["error"] = str(exc)[:1000]

    manual_status_rows.append(status)

manual_features_df = pd.DataFrame(manual_feature_rows).fillna(0)
manual_status_df = pd.DataFrame(manual_status_rows)

manual_features_df.to_csv(
    MANUAL_FEATURE_DIR / "features_manual_0410.csv",
    index=False
)
manual_status_df.to_csv(
    MANUAL_STATUS_DIR / "status_manual_0410.csv",
    index=False
)

print("=== KẾT QUẢ ===")
print(manual_status_df["status"].value_counts())
print("Số feature rows:", len(manual_features_df))
print("Số contract IDs duy nhất:", manual_features_df["contract_id"].nunique())
print("Đã lưu features:", MANUAL_FEATURE_DIR / "features_manual_0410.csv")
print("Đã lưu status:", MANUAL_STATUS_DIR / "status_manual_0410.csv")


Trích xuất opcode features:   0%|          | 0/10 [00:00<?, ?it/s]

=== KẾT QUẢ ===
status
FEATURE_SUCCESS    10
Name: count, dtype: int64
Số feature rows: 10
Số contract IDs duy nhất: 10
Đã lưu features: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/feature_batches_manual_0410/features_manual_0410.csv
Đã lưu status: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/status_batches_manual_0410/status_manual_0410.csv


In [ ]:
manual_features_df = pd.read_csv(
    MANUAL_FEATURE_DIR / "features_manual_0410.csv"
)
manual_status_df = pd.read_csv(
    MANUAL_STATUS_DIR / "status_manual_0410.csv"
)

existing_ids = set(combined_features_df["contract_id"].astype(str))
manual_ids = set(manual_features_df["contract_id"].astype(str))

overlap_ids = existing_ids & manual_ids

print("Features hiện tại:", len(combined_features_df))
print("Features mới:", len(manual_features_df))
print("ID bị trùng:", len(overlap_ids))

if overlap_ids:
    print("Các ID trùng:", sorted(overlap_ids))
    raise ValueError(
        "Có ID đã tồn tại. Dừng để tránh cộng features trùng."
    )

combined_features_df["contract_id"] = (
    combined_features_df["contract_id"].astype(str)
)
manual_features_df["contract_id"] = (
    manual_features_df["contract_id"].astype(str)
)

combined_features_df = pd.concat(
    [combined_features_df, manual_features_df],
    ignore_index=True,
    sort=False
)

opcode_cols = [
    col for col in combined_features_df.columns
    if col != "contract_id"
]

combined_features_df[opcode_cols] = (
    combined_features_df[opcode_cols].fillna(0)
)

manual_ids_list = manual_status_df["contract_id"].astype(str).tolist()

final_status_df["contract_id"] = (
    final_status_df["contract_id"].astype(str)
)

final_status_df = final_status_df[
    ~final_status_df["contract_id"].isin(manual_ids_list)
]

final_status_df = pd.concat(
    [final_status_df, manual_status_df],
    ignore_index=True,
    sort=False
)

print("\n=== KIỂM TRA SAU KHI HỢP NHẤT ===")
print("Tổng feature rows:", len(combined_features_df))
print("Feature IDs duy nhất:", combined_features_df["contract_id"].nunique())
print("Tổng status rows:", len(final_status_df))
print("Status IDs duy nhất:", final_status_df["contract_id"].nunique())
print("\nPhân bố trạng thái:")
print(final_status_df["status"].value_counts())

assert combined_features_df["contract_id"].is_unique
assert final_status_df["contract_id"].is_unique


Features hiện tại: 41218
Features mới: 10
ID bị trùng: 0

=== KIỂM TRA SAU KHI HỢP NHẤT ===
Tổng feature rows: 41228
Feature IDs duy nhất: 41228
Tổng status rows: 43232
Status IDs duy nhất: 43232

Phân bố trạng thái:
status
FEATURE_SUCCESS          41228
NO_COMPILER_CANDIDATE     1235
PROCESS_FAILED             764
COMPILE_NO_RUNTIME           5
Name: count, dtype: int64


In [ ]:
CHECKPOINT_DIR = (
    Path("/content/drive/MyDrive/IS6002/merged_dataset/opcode_features")
    / "checkpoints"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

features_checkpoint = CHECKPOINT_DIR / "combined_features_41228.csv"
status_checkpoint = CHECKPOINT_DIR / "final_status_43232.csv"

combined_features_df.to_csv(features_checkpoint, index=False)
final_status_df.to_csv(status_checkpoint, index=False)

print("Đã lưu checkpoint:")
print(features_checkpoint)
print(status_checkpoint)

print("\nKiểm tra:")
print("Features:", len(combined_features_df),
      "| ID duy nhất:", combined_features_df["contract_id"].nunique())
print("Status:", len(final_status_df),
      "| ID duy nhất:", final_status_df["contract_id"].nunique())


Đã lưu checkpoint:
/content/drive/MyDrive/IS6002/merged_dataset/opcode_features/checkpoints/combined_features_41228.csv
/content/drive/MyDrive/IS6002/merged_dataset/opcode_features/checkpoints/final_status_43232.csv

Kiểm tra:
Features: 41228 | ID duy nhất: 41228
Status: 43232 | ID duy nhất: 43232


In [ ]:

base = "https://raw.githubusercontent.com/argotorg/solc-bin/gh-pages"

for folder in ["emscripten-asmjs", "emscripten-wasm32", "bin"]:
    url = f"{base}/{folder}/list.json"
    try:
        r = requests.get(url, timeout=30)
        print(f"\n[{folder}] HTTP {r.status_code}")

        if not r.ok:
            continue

        data = r.json()
        releases = data.get("releases", {})

        for version in ["0.4.4", "0.4.6", "0.4.8", "0.4.9"]:
            print(f"{version}: {releases.get(version, 'KHÔNG CÓ')}")
    except Exception as e:
        print(type(e).__name__, str(e)[:200])



[emscripten-asmjs] HTTP 200
0.4.4: solc-emscripten-asmjs-v0.4.4+commit.4633f3de.js
0.4.6: solc-emscripten-asmjs-v0.4.6+commit.2dabbdf0.js
0.4.8: solc-emscripten-asmjs-v0.4.8+commit.60cc1668.js
0.4.9: solc-emscripten-asmjs-v0.4.9+commit.364da425.js

[emscripten-wasm32] HTTP 200
0.4.4: solc-emscripten-wasm32-v0.4.4+commit.4633f3de.js
0.4.6: solc-emscripten-wasm32-v0.4.6+commit.2dabbdf0.js
0.4.8: solc-emscripten-wasm32-v0.4.8+commit.60cc1668.js
0.4.9: solc-emscripten-wasm32-v0.4.9+commit.364da425.js

[bin] HTTP 200
0.4.4: soljson-v0.4.4+commit.4633f3de.js
0.4.6: soljson-v0.4.6+commit.2dabbdf0.js
0.4.8: soljson-v0.4.8+commit.60cc1668.js
0.4.9: soljson-v0.4.9+commit.364da425.js


In [ ]:

import pandas as pd
from pathlib import Path


ids_048 = remaining_pragma_df[
    remaining_pragma_df["pragmas"].astype(str).str.strip().eq("0.4.8")
]["contract_id"].astype(str).tolist()

print("Số contract pragma 0.4.8:", len(ids_048))
print("Một vài ID:", ids_048[:10])

if ids_048:
    test_id = ids_048[0]
    test_path = CONTRACTS_DIR / f"{test_id}.sol"

    print("\nContract thử:", test_id)
    print("File tồn tại:", test_path.exists())

    if test_path.exists():
        source = test_path.read_text(errors="replace")
        print("\nĐầu mã nguồn:")
        print(source[:1000])


Số contract pragma 0.4.8: 27
Một vài ID: ['C002244', 'C002346', 'C005508', 'C006248', 'C007219', 'C007486', 'C007999', 'C009012', 'C011950', 'C013430']

Contract thử: C002244
File tồn tại: True

Đầu mã nguồn:
pragma solidity 0.4.8;
contract ERC20TokenInterface {
    function totalSupply() constant returns (uint256 supply);
    function balanceOf(address _owner) constant public returns (uint256 balance);
    function transfer(address _to, uint256 _value) public returns (bool success);
    function transferFrom(address _from, address _to, uint256 _value) public returns (bool success);
    function approve(address _spender, uint256 _value) public returns (bool success);
    function allowance(address _owner, address _spender) constant public returns (uint256 remaining);

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);
}

contract Gidnist is ERC20TokenInterface {
  string public 

In [ ]:
test_id = "C002244"
source_path = CONTRACTS_DIR / f"{test_id}.sol"
source = source_path.read_text(errors="replace")

test_script = Path("/content/solc_legacy/compile_file_048.js")

test_script.write_text(r'''
const fs = require("fs");
const solc = require("./soljson-v0.4.8.js");

const sourcePath = process.argv[2];
const source = fs.readFileSync(sourcePath, "utf8");

const compile = solc.cwrap(
    "compileJSON",
    "string",
    ["string", "number"]
);

const output = JSON.parse(compile(source, 0));

if (output.errors) {
    console.log("Compiler messages:");
    for (const error of output.errors) {
        console.log(error);
    }
}

const contracts = output.contracts || {};
const result = {};

for (const [name, contract] of Object.entries(contracts)) {
    result[name] = {
        runtimeBytecode: contract.runtimeBytecode || "",
        creationBytecode: contract.bytecode || ""
    };
}

console.log("RESULT_JSON:" + JSON.stringify(result));
''')

result = subprocess.run(
    ["node", str(test_script), str(source_path)],
    cwd="/content/solc_legacy",
    capture_output=True,
    text=True,
    timeout=120
)

print("Exit code:", result.returncode)
print("STDERR:", result.stderr[:1000])

result_line = next(
    (line for line in result.stdout.splitlines()
     if line.startswith("RESULT_JSON:")),
    None
)

if result.returncode != 0 or result_line is None:
    print(result.stdout[:3000])
    raise RuntimeError("Compiler chưa trả về kết quả JSON hợp lệ.")

contracts_048_test = json.loads(result_line[len("RESULT_JSON:"):])

print("\nSố contract được biên dịch:", len(contracts_048_test))

for name, artifact in contracts_048_test.items():
    runtime_hex = artifact["runtimeBytecode"]
    print(
        name,
        "| runtime bytecode:", len(runtime_hex) // 2, "bytes",
        "| có bytecode:", bool(runtime_hex)
    )


Exit code: 0
STDERR: (node:198983) V8: /content/solc_legacy/soljson-v0.4.8.js:3 Invalid asm.js: Invalid member of stdlib
(Use `node --trace-warnings ...` to show where the warning was created)


Số contract được biên dịch: 2
ERC20TokenInterface | runtime bytecode: 0 bytes | có bytecode: False
Gidnist | runtime bytecode: 4531 bytes | có bytecode: True


In [ ]:

test_runtime_hex = contracts_048_test["Gidnist"]["runtimeBytecode"].removeprefix("0x")

metadata_result = strip_solidity_metadata(test_runtime_hex)

print("Kiểu dữ liệu:", type(metadata_result))
print("Số phần tử:", len(metadata_result) if isinstance(metadata_result, tuple) else "Không phải tuple")

if isinstance(metadata_result, tuple):
    for i, item in enumerate(metadata_result):
        print(f"\nPhần tử [{i}]")
        print("Kiểu:", type(item))
        if isinstance(item, str):
            print("Độ dài:", len(item))
            print("Đầu:", item[:100])
        else:
            print("Giá trị:", repr(item)[:300])
else:
    print("Giá trị:", repr(metadata_result)[:500])


Kiểu dữ liệu: <class 'tuple'>
Số phần tử: 3

Phần tử [0]
Kiểu: <class 'str'>
Độ dài: 8976
Đầu: 606060405236156100ef576000357c0100000000000000000000000000000000000000000000000000000000900463ffffff

Phần tử [1]
Kiểu: <class 'bool'>
Giá trị: True

Phần tử [2]
Kiểu: <class 'str'>
Độ dài: 13
Đầu: CBOR_VERIFIED


In [ ]:

test_runtime_hex = (
    contracts_048_test["Gidnist"]["runtimeBytecode"]
    .removeprefix("0x")
)

runtime_without_metadata, metadata_verified, metadata_status = (
    strip_solidity_metadata(test_runtime_hex)
)

opcode_counts_test = extract_opcode_counts(runtime_without_metadata)

print("=== KẾT QUẢ KIỂM TRA ===")
print("Runtime ban đầu:", len(test_runtime_hex) // 2, "bytes")
print("Runtime sau xử lý metadata:", len(runtime_without_metadata) // 2, "bytes")
print("Metadata verified:", metadata_verified)
print("Metadata status:", metadata_status)
print("Số loại opcode:", len(opcode_counts_test))
print("Tổng số opcode:", sum(opcode_counts_test.values()))
print("\nMột số opcode:")
print(dict(list(opcode_counts_test.items())[:20]))

assert runtime_without_metadata
assert opcode_counts_test
assert all(
    isinstance(v, int) and v >= 0
    for v in opcode_counts_test.values()
)


=== KẾT QUẢ KIỂM TRA ===
Runtime ban đầu: 4531 bytes
Runtime sau xử lý metadata: 4488 bytes
Metadata verified: True
Metadata status: CBOR_VERIFIED
Số loại opcode: 48
Tổng số opcode: 2315

Một số opcode:
{'PUSH1': 329, 'MSTORE': 92, 'CALLDATASIZE': 1, 'ISZERO': 46, 'PUSH2': 128, 'JUMPI': 63, 'CALLDATALOAD': 20, 'PUSH29': 1, 'SWAP1': 180, 'DIV': 11, 'PUSH4': 18, 'AND': 82, 'DUP1': 147, 'EQ': 25, 'JUMPDEST': 95, 'JUMP': 51, 'CALLVALUE': 17, 'MLOAD': 66, 'ADD': 135, 'DUP3': 69}


In [ ]:
legacy_script = Path("/content/solc_legacy/compile_file_legacy.js")

legacy_script.write_text(r'''
const fs = require("fs");
const solc = require("./soljson-v0.4.8.js");

const sourcePath = process.argv[2];
const source = fs.readFileSync(sourcePath, "utf8");

const compile = solc.cwrap(
    "compileJSON",
    "string",
    ["string", "number"]
);

const output = JSON.parse(compile(source, 0));

const result = {
    errors: output.errors || [],
    contracts: {}
};

for (const [name, artifact] of Object.entries(output.contracts || {})) {
    result.contracts[name] = {
        runtimeBytecode: artifact.runtimeBytecode || ""
    };
}

console.log("RESULT_JSON:" + JSON.stringify(result));
''')

print("Script:", legacy_script)
print("Tồn tại:", legacy_script.exists())
print("Dung lượng:", legacy_script.stat().st_size, "bytes")
print("\nĐầu script:")
print(legacy_script.read_text()[:300])


Script: /content/solc_legacy/compile_file_legacy.js
Tồn tại: True
Dung lượng: 596 bytes

Đầu script:

const fs = require("fs");
const solc = require("./soljson-v0.4.8.js");

const sourcePath = process.argv[2];
const source = fs.readFileSync(sourcePath, "utf8");

const compile = solc.cwrap(
    "compileJSON",
    "string",
    ["string", "number"]
);

const output = JSON.parse(compile(source, 0));




In [ ]:
test_id = "C002244"
source_path = CONTRACTS_DIR / f"{test_id}.sol"

result = subprocess.run(
    [
        "node",
        "/content/solc_legacy/compile_file_legacy.js",
        str(source_path)
    ],
    cwd="/content/solc_legacy",
    capture_output=True,
    text=True,
    timeout=120
)

print("Exit code:", result.returncode)
print("STDERR:", result.stderr[:500])

result_line = next(
    (
        line for line in result.stdout.splitlines()
        if line.startswith("RESULT_JSON:")
    ),
    None
)

if result.returncode != 0 or result_line is None:
    print("STDOUT:", result.stdout[:2000])
    raise RuntimeError("Script chưa trả về JSON hợp lệ.")

compiled_test = json.loads(result_line[len("RESULT_JSON:"):])

print("\nTên contract:", list(compiled_test["contracts"].keys()))
for name, artifact in compiled_test["contracts"].items():
    print(
        name,
        "| runtime bytes:",
        len(artifact["runtimeBytecode"]) // 2
    )

assert "Gidnist" in compiled_test["contracts"]
assert compiled_test["contracts"]["Gidnist"]["runtimeBytecode"]


Exit code: 0
STDERR: (node:199610) V8: /content/solc_legacy/soljson-v0.4.8.js:3 Invalid asm.js: Invalid member of stdlib
(Use `node --trace-warnings ...` to show where the warning was created)


Tên contract: ['ERC20TokenInterface', 'Gidnist']
ERC20TokenInterface | runtime bytes: 0
Gidnist | runtime bytes: 4531


In [ ]:
LEGACY_048_DIR = Path(
    "/content/drive/MyDrive/IS6002/merged_dataset/opcode_features"
) / "legacy_048_test"

FEATURE_DIR_048 = LEGACY_048_DIR / "features"
STATUS_DIR_048 = LEGACY_048_DIR / "status"

FEATURE_DIR_048.mkdir(parents=True, exist_ok=True)
STATUS_DIR_048.mkdir(parents=True, exist_ok=True)

feature_rows_048 = []
status_rows_048 = []

for contract_id in tqdm(ids_048, desc="Compile Solidity 0.4.8"):
    source_path = CONTRACTS_DIR / f"{contract_id}.sol"

    try:
        result = subprocess.run(
            [
                "node",
                "/content/solc_legacy/compile_file_legacy.js",
                str(source_path)
            ],
            cwd="/content/solc_legacy",
            capture_output=True,
            text=True,
            timeout=120
        )

        result_line = next(
            (
                line for line in result.stdout.splitlines()
                if line.startswith("RESULT_JSON:")
            ),
            None
        )

        if result.returncode != 0 or result_line is None:
            status_rows_048.append({
                "contract_id": contract_id,
                "status": "PROCESS_FAILED",
                "details": (result.stderr or result.stdout)[:1000]
            })
            continue

        compiled = json.loads(result_line[len("RESULT_JSON:"):])
        file_counts = Counter()
        runtime_contract_count = 0
        metadata_verified_count = 0

        for name, artifact in compiled["contracts"].items():
            runtime_hex = artifact.get("runtimeBytecode", "").removeprefix("0x")

            if not runtime_hex:
                continue

            runtime_contract_count += 1
            stripped_hex, verified, metadata_status = (
                strip_solidity_metadata(runtime_hex)
            )

            if verified:
                metadata_verified_count += 1

            counts = extract_opcode_counts(stripped_hex)
            file_counts.update(counts)

        if runtime_contract_count == 0 or not file_counts:
            status_rows_048.append({
                "contract_id": contract_id,
                "status": "COMPILE_NO_RUNTIME",
                "details": "Không có runtime bytecode triển khai để đếm opcode"
            })
            continue

        feature_rows_048.append({
            "contract_id": contract_id,
            **dict(file_counts)
        })

        status_rows_048.append({
            "contract_id": contract_id,
            "status": "FEATURE_SUCCESS",
            "details": (
                f"runtime_contracts={runtime_contract_count}; "
                f"metadata_verified={metadata_verified_count}"
            )
        })

    except Exception as e:
        status_rows_048.append({
            "contract_id": contract_id,
            "status": "PROCESS_FAILED",
            "details": f"{type(e).__name__}: {e}"[:1000]
        })

features_048_df = pd.DataFrame(feature_rows_048).fillna(0)
status_048_df = pd.DataFrame(status_rows_048)

features_048_df.to_csv(
    FEATURE_DIR_048 / "features_048.csv", index=False
)
status_048_df.to_csv(
    STATUS_DIR_048 / "status_048.csv", index=False
)

print("\n=== KẾT QUẢ BATCH 0.4.8 ===")
print("Số contract đầu vào:", len(ids_048))
print("Feature rows:", len(features_048_df))
print("Feature ID duy nhất:", features_048_df["contract_id"].nunique()
      if not features_048_df.empty else 0)
print("\nPhân bố trạng thái:")
print(status_048_df["status"].value_counts(dropna=False))
print("\nĐã lưu tại:", LEGACY_048_DIR)

assert status_048_df["contract_id"].nunique() == len(ids_048)
if not features_048_df.empty:
    assert features_048_df["contract_id"].is_unique


Compile Solidity 0.4.8:   0%|          | 0/27 [00:00<?, ?it/s]


=== KẾT QUẢ BATCH 0.4.8 ===
Số contract đầu vào: 27
Feature rows: 27
Feature ID duy nhất: 27

Phân bố trạng thái:
status
FEATURE_SUCCESS    27
Name: count, dtype: int64

Đã lưu tại: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/legacy_048_test


In [ ]:
features_048_df = pd.read_csv(FEATURE_DIR_048 / "features_048.csv")
status_048_df = pd.read_csv(STATUS_DIR_048 / "status_048.csv")

features_048_df["contract_id"] = features_048_df["contract_id"].astype(str)
status_048_df["contract_id"] = status_048_df["contract_id"].astype(str)

combined_features_df["contract_id"] = combined_features_df["contract_id"].astype(str)
final_status_df["contract_id"] = final_status_df["contract_id"].astype(str)

overlap = set(combined_features_df["contract_id"]) & set(features_048_df["contract_id"])
print("Feature ID trùng:", len(overlap))

if overlap:
    raise ValueError(f"Dừng hợp nhất vì ID đã tồn tại: {sorted(overlap)}")

combined_features_df = pd.concat(
    [combined_features_df, features_048_df],
    ignore_index=True,
    sort=False
)

opcode_cols = [
    col for col in combined_features_df.columns
    if col != "contract_id"
]
combined_features_df[opcode_cols] = combined_features_df[opcode_cols].fillna(0)

ids_048_set = set(status_048_df["contract_id"])
final_status_df = final_status_df[
    ~final_status_df["contract_id"].isin(ids_048_set)
]
final_status_df = pd.concat(
    [final_status_df, status_048_df],
    ignore_index=True,
    sort=False
)

print("\n=== SAU KHI HỢP NHẤT ===")
print("Feature rows:", len(combined_features_df))
print("Feature IDs duy nhất:", combined_features_df["contract_id"].nunique())
print("Status rows:", len(final_status_df))
print("Status IDs duy nhất:", final_status_df["contract_id"].nunique())
print("\nPhân bố trạng thái:")
print(final_status_df["status"].value_counts())

assert combined_features_df["contract_id"].is_unique
assert final_status_df["contract_id"].is_unique
assert len(final_status_df) == 43232

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
combined_features_df.to_csv(
    CHECKPOINT_DIR / "combined_features_41255.csv", index=False
)
final_status_df.to_csv(
    CHECKPOINT_DIR / "final_status_43232_v2.csv", index=False
)

print("\nĐã lưu checkpoint mới.")


Feature ID trùng: 0

=== SAU KHI HỢP NHẤT ===
Feature rows: 41255
Feature IDs duy nhất: 41255
Status rows: 43232
Status IDs duy nhất: 43232

Phân bố trạng thái:
status
FEATURE_SUCCESS          41255
NO_COMPILER_CANDIDATE     1208
PROCESS_FAILED             764
COMPILE_NO_RUNTIME           5
Name: count, dtype: int64

Đã lưu checkpoint mới.


In [ ]:
COMPILER_DIR_049 = Path("/content/solc_legacy/solc_049")
COMPILER_DIR_049.mkdir(parents=True, exist_ok=True)

url_049 = (
    "https://raw.githubusercontent.com/argotorg/solc-bin/"
    "gh-pages/bin/soljson-v0.4.9+commit.364da425.js"
)

compiler_path_049 = COMPILER_DIR_049 / "soljson-v0.4.9.js"

r = requests.get(url_049, timeout=60)
r.raise_for_status()
compiler_path_049.write_bytes(r.content)

print("Compiler:", compiler_path_049)
print("Dung lượng:", compiler_path_049.stat().st_size, "bytes")
assert compiler_path_049.stat().st_size > 1_000_000


Compiler: /content/solc_legacy/solc_049/soljson-v0.4.9.js
Dung lượng: 8080899 bytes


In [ ]:
LEGACY_049_DIR = (
    Path("/content/drive/MyDrive/IS6002/merged_dataset/opcode_features")
    / "legacy_049"
)
FEATURE_DIR_049 = LEGACY_049_DIR / "features"
STATUS_DIR_049 = LEGACY_049_DIR / "status"

FEATURE_DIR_049.mkdir(parents=True, exist_ok=True)
STATUS_DIR_049.mkdir(parents=True, exist_ok=True)

COMPILER_DIR_049 = Path("/content/solc_legacy/solc_049")

feature_rows_049 = []
status_rows_049 = []

for contract_id in ids_049:
    source_path = CONTRACTS_DIR / f"{contract_id}.sol"

    try:
        result = subprocess.run(
            [
                "node",
                str(COMPILER_DIR_049 / "compile_file_049.js"),
                str(source_path)
            ],
            cwd=str(COMPILER_DIR_049),
            capture_output=True,
            text=True,
            timeout=120
        )

        result_line = next(
            (
                line for line in result.stdout.splitlines()
                if line.startswith("RESULT_JSON:")
            ),
            None
        )

        if result.returncode != 0 or result_line is None:
            raise RuntimeError(
                f"Compile failed: {result.stderr[-500:]}"
            )

        compiled = json.loads(
            result_line[len("RESULT_JSON:"):]
        )

        total_counts = Counter()
        runtime_contracts = 0

        for name, artifact in compiled["contracts"].items():
            runtime_hex = artifact.get("runtimeBytecode", "")

            if not runtime_hex:
                continue

            stripped_hex, verified, metadata_status = (
                strip_solidity_metadata(runtime_hex)
            )

            total_counts.update(
                extract_opcode_counts(stripped_hex)
            )
            runtime_contracts += 1

        if runtime_contracts == 0:
            status = "COMPILE_NO_RUNTIME"
        else:
            feature_rows_049.append({
                "contract_id": contract_id,
                **dict(total_counts)
            })
            status = "FEATURE_SUCCESS"

        status_rows_049.append({
            "contract_id": contract_id,
            "status": status,
            "runtime_contracts": runtime_contracts,
            "error": ""
        })

        print(
            contract_id,
            "|", status,
            "| runtime contracts:", runtime_contracts,
            "| opcode kinds:", len(total_counts)
        )

    except Exception as e:
        status_rows_049.append({
            "contract_id": contract_id,
            "status": "PROCESS_FAILED",
            "runtime_contracts": 0,
            "error": str(e)[:1000]
        })
        print(contract_id, "| PROCESS_FAILED |", str(e)[:250])

features_049_df = pd.DataFrame(feature_rows_049)
status_049_df = pd.DataFrame(status_rows_049)

features_049_df.to_csv(
    FEATURE_DIR_049 / "features_049.csv",
    index=False
)
status_049_df.to_csv(
    STATUS_DIR_049 / "status_049.csv",
    index=False
)

print("\n=== TỔNG KẾT SOLC 0.4.9 ===")
print("Feature rows:", len(features_049_df))
print(status_049_df["status"].value_counts().to_dict())
print("Feature file:", FEATURE_DIR_049 / "features_049.csv")
print("Status file:", STATUS_DIR_049 / "status_049.csv")


C000359 | FEATURE_SUCCESS | runtime contracts: 3 | opcode kinds: 62
C024258 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 57

=== TỔNG KẾT SOLC 0.4.9 ===
Feature rows: 2
{'FEATURE_SUCCESS': 2}
Feature file: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/legacy_049/features/features_049.csv
Status file: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/legacy_049/status/status_049.csv


In [ ]:
ids_features_049 = set(features_049_df["contract_id"].astype(str))
existing_ids = set(combined_features_df["contract_id"].astype(str))

overlap = ids_features_049 & existing_ids

print("Features 0.4.9:", len(features_049_df))
print("ID bị trùng với dữ liệu hiện tại:", overlap)

if overlap:
    raise ValueError("Có contract ID bị trùng, dừng gộp để tránh nhân đôi.")

combined_features_df = pd.concat(
    [combined_features_df, features_049_df],
    ignore_index=True,
    sort=False
).fillna(0)

combined_features_df = combined_features_df.loc[
    ~combined_features_df["contract_id"].astype(str).duplicated()
].reset_index(drop=True)

status_lookup_049 = status_049_df.set_index("contract_id")["status"].to_dict()

final_status_df["status"] = final_status_df.apply(
    lambda row: status_lookup_049.get(
        str(row["contract_id"]), row["status"]
    ),
    axis=1
)

print("\n=== SAU KHI GỘP ===")
print("Unique feature IDs:", combined_features_df["contract_id"].nunique())
print("Feature rows:", len(combined_features_df))
print("Unique status IDs:", final_status_df["contract_id"].nunique())
print(final_status_df["status"].value_counts().to_dict())

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

combined_features_df.to_csv(
    CHECKPOINT_DIR / "combined_features_41257.csv",
    index=False
)

final_status_df.to_csv(
    CHECKPOINT_DIR / "final_status_43232_v3.csv",
    index=False
)

print("\nĐã lưu checkpoint:")
print(CHECKPOINT_DIR / "combined_features_41257.csv")
print(CHECKPOINT_DIR / "final_status_43232_v3.csv")


Features 0.4.9: 2
ID bị trùng với dữ liệu hiện tại: set()

=== SAU KHI GỘP ===
Unique feature IDs: 41257
Feature rows: 41257
Unique status IDs: 43232
{'FEATURE_SUCCESS': 41257, 'NO_COMPILER_CANDIDATE': 1206, 'PROCESS_FAILED': 764, 'COMPILE_NO_RUNTIME': 5}

Đã lưu checkpoint:
/content/drive/MyDrive/IS6002/merged_dataset/opcode_features/checkpoints/combined_features_41257.csv
/content/drive/MyDrive/IS6002/merged_dataset/opcode_features/checkpoints/final_status_43232_v3.csv


In [ ]:

import json
import subprocess
from pathlib import Path
from collections import Counter
import pandas as pd

LEGACY_046_DIR = (
    Path("/content/drive/MyDrive/IS6002/merged_dataset/opcode_features")
    / "legacy_046"
)
FEATURE_DIR_046 = LEGACY_046_DIR / "features"
STATUS_DIR_046 = LEGACY_046_DIR / "status"

FEATURE_DIR_046.mkdir(parents=True, exist_ok=True)
STATUS_DIR_046.mkdir(parents=True, exist_ok=True)

feature_rows_046 = []
status_rows_046 = []

for contract_id in ids_046:
    source_path = CONTRACTS_DIR / f"{contract_id}.sol"

    try:
        result = subprocess.run(
            [
                "node",
                str(COMPILER_DIR_046 / "compile_file_046.js"),
                str(source_path)
            ],
            cwd=str(COMPILER_DIR_046),
            capture_output=True,
            text=True,
            timeout=120
        )

        result_line = next(
            (
                line for line in result.stdout.splitlines()
                if line.startswith("RESULT_JSON:")
            ),
            None
        )

        if result.returncode != 0 or result_line is None:
            raise RuntimeError(
                f"Compile failed: {result.stderr[-500:]}"
            )

        compiled = json.loads(
            result_line[len("RESULT_JSON:"):]
        )

        total_counts = Counter()
        runtime_contracts = 0

        for name, artifact in compiled["contracts"].items():
            runtime_hex = artifact.get("runtimeBytecode", "")

            if not runtime_hex:
                continue

            stripped_hex, verified, metadata_status = (
                strip_solidity_metadata(runtime_hex)
            )

            total_counts.update(
                extract_opcode_counts(stripped_hex)
            )
            runtime_contracts += 1

        if runtime_contracts == 0:
            status = "COMPILE_NO_RUNTIME"
        else:
            feature_rows_046.append({
                "contract_id": contract_id,
                **dict(total_counts)
            })
            status = "FEATURE_SUCCESS"

        status_rows_046.append({
            "contract_id": contract_id,
            "status": status,
            "runtime_contracts": runtime_contracts,
            "error": ""
        })

        print(
            contract_id,
            "|", status,
            "| runtime contracts:", runtime_contracts,
            "| opcode kinds:", len(total_counts)
        )

    except Exception as e:
        status_rows_046.append({
            "contract_id": contract_id,
            "status": "PROCESS_FAILED",
            "runtime_contracts": 0,
            "error": str(e)[:1000]
        })
        print(contract_id, "| PROCESS_FAILED |", str(e)[:250])

features_046_df = pd.DataFrame(feature_rows_046)
status_046_df = pd.DataFrame(status_rows_046)

features_046_df.to_csv(
    FEATURE_DIR_046 / "features_046.csv",
    index=False
)
status_046_df.to_csv(
    STATUS_DIR_046 / "status_046.csv",
    index=False
)

print("\n=== TỔNG KẾT SOLC 0.4.6 ===")
print("Feature rows:", len(features_046_df))
print(status_046_df["status"].value_counts().to_dict())
print("Feature file:", FEATURE_DIR_046 / "features_046.csv")
print("Status file:", STATUS_DIR_046 / "status_046.csv")


C019576 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 54
C042523 | FEATURE_SUCCESS | runtime contracts: 2 | opcode kinds: 60

=== TỔNG KẾT SOLC 0.4.6 ===
Feature rows: 2
{'FEATURE_SUCCESS': 2}
Feature file: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/legacy_046/features/features_046.csv
Status file: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/legacy_046/status/status_046.csv


In [ ]:
ids_features_046 = set(features_046_df["contract_id"].astype(str))
existing_ids = set(combined_features_df["contract_id"].astype(str))

overlap_046 = ids_features_046 & existing_ids

print("Features 0.4.6:", len(features_046_df))
print("ID bị trùng:", overlap_046)

if overlap_046:
    raise ValueError("Có contract ID bị trùng, dừng gộp.")

combined_features_df = pd.concat(
    [combined_features_df, features_046_df],
    ignore_index=True,
    sort=False
).fillna(0)

combined_features_df = combined_features_df.loc[
    ~combined_features_df["contract_id"].astype(str).duplicated()
].reset_index(drop=True)

status_lookup_046 = status_046_df.set_index("contract_id")["status"].to_dict()

final_status_df["status"] = final_status_df.apply(
    lambda row: status_lookup_046.get(
        str(row["contract_id"]), row["status"]
    ),
    axis=1
)

print("\n=== SAU KHI GỘP ===")
print("Unique feature IDs:", combined_features_df["contract_id"].nunique())
print("Feature rows:", len(combined_features_df))
print("Unique status IDs:", final_status_df["contract_id"].nunique())
print(final_status_df["status"].value_counts().to_dict())

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

combined_features_df.to_csv(
    CHECKPOINT_DIR / "combined_features_41259.csv",
    index=False
)

final_status_df.to_csv(
    CHECKPOINT_DIR / "final_status_43232_v4.csv",
    index=False
)

print("\nĐã lưu checkpoint:")
print(CHECKPOINT_DIR / "combined_features_41259.csv")
print(CHECKPOINT_DIR / "final_status_43232_v4.csv")


Features 0.4.6: 2
ID bị trùng: set()

=== SAU KHI GỘP ===
Unique feature IDs: 41259
Feature rows: 41259
Unique status IDs: 43232
{'FEATURE_SUCCESS': 41259, 'NO_COMPILER_CANDIDATE': 1204, 'PROCESS_FAILED': 764, 'COMPILE_NO_RUNTIME': 5}

Đã lưu checkpoint:
/content/drive/MyDrive/IS6002/merged_dataset/opcode_features/checkpoints/combined_features_41259.csv
/content/drive/MyDrive/IS6002/merged_dataset/opcode_features/checkpoints/final_status_43232_v4.csv


In [ ]:
LEGACY_044_DIR = (
    Path("/content/drive/MyDrive/IS6002/merged_dataset/opcode_features")
    / "legacy_044"
)
FEATURE_DIR_044 = LEGACY_044_DIR / "features"
STATUS_DIR_044 = LEGACY_044_DIR / "status"

FEATURE_DIR_044.mkdir(parents=True, exist_ok=True)
STATUS_DIR_044.mkdir(parents=True, exist_ok=True)

feature_rows_044 = []
status_rows_044 = []

for contract_id in ids_044:
    source_path = CONTRACTS_DIR / f"{contract_id}.sol"

    try:
        result = subprocess.run(
            [
                "node",
                str(COMPILER_DIR_044 / "compile_file_044.js"),
                str(source_path)
            ],
            cwd=str(COMPILER_DIR_044),
            capture_output=True,
            text=True,
            timeout=120
        )

        result_line = next(
            (
                line for line in result.stdout.splitlines()
                if line.startswith("RESULT_JSON:")
            ),
            None
        )

        if result.returncode != 0 or result_line is None:
            raise RuntimeError(
                f"Compile failed: {result.stderr[-500:]}"
            )

        compiled = json.loads(
            result_line[len("RESULT_JSON:"):]
        )

        total_counts = Counter()
        runtime_contracts = 0

        for name, artifact in compiled["contracts"].items():
            runtime_hex = artifact.get("runtimeBytecode", "")

            if not runtime_hex:
                continue

            stripped_hex, verified, metadata_status = (
                strip_solidity_metadata(runtime_hex)
            )

            total_counts.update(
                extract_opcode_counts(stripped_hex)
            )
            runtime_contracts += 1

        if runtime_contracts == 0:
            status = "COMPILE_NO_RUNTIME"
        else:
            feature_rows_044.append({
                "contract_id": contract_id,
                **dict(total_counts)
            })
            status = "FEATURE_SUCCESS"

        status_rows_044.append({
            "contract_id": contract_id,
            "status": status,
            "runtime_contracts": runtime_contracts,
            "error": ""
        })

        print(
            contract_id,
            "|", status,
            "| runtime contracts:", runtime_contracts,
            "| opcode kinds:", len(total_counts)
        )

    except Exception as e:
        status_rows_044.append({
            "contract_id": contract_id,
            "status": "PROCESS_FAILED",
            "runtime_contracts": 0,
            "error": str(e)[:1000]
        })
        print(contract_id, "| PROCESS_FAILED |", str(e)[:250])

features_044_df = pd.DataFrame(feature_rows_044)
status_044_df = pd.DataFrame(status_rows_044)

features_044_df.to_csv(
    FEATURE_DIR_044 / "features_044.csv",
    index=False
)
status_044_df.to_csv(
    STATUS_DIR_044 / "status_044.csv",
    index=False
)

print("\n=== TỔNG KẾT SOLC 0.4.4 ===")
print("Số contract đầu vào:", len(ids_044))
print("Feature rows:", len(features_044_df))
print(status_044_df["status"].value_counts().to_dict())
print("Feature file:", FEATURE_DIR_044 / "features_044.csv")
print("Status file:", STATUS_DIR_044 / "status_044.csv")


C004021 | FEATURE_SUCCESS | runtime contracts: 2 | opcode kinds: 60
C006113 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 57
C008403 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 44
C010687 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 43
C016360 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 79
C029334 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 46
C029790 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 79
C030975 | FEATURE_SUCCESS | runtime contracts: 2 | opcode kinds: 59
C033198 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 57
C033564 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 25
C035663 | FEATURE_SUCCESS | runtime contracts: 2 | opcode kinds: 59
C035665 | FEATURE_SUCCESS | runtime contracts: 1 | opcode kinds: 41

=== TỔNG KẾT SOLC 0.4.4 ===
Số contract đầu vào: 12
Feature rows: 12
{'FEATURE_SUCCESS': 12}
Feature file: /content/drive/MyDrive/IS6002/merged_dataset/opcode_features/legacy_044/feat

In [ ]:
ids_features_044 = set(features_044_df["contract_id"].astype(str))
existing_ids = set(combined_features_df["contract_id"].astype(str))

overlap_044 = ids_features_044 & existing_ids
print("Features 0.4.4:", len(features_044_df))
print("ID bị trùng:", overlap_044)

if overlap_044:
    raise ValueError("Có contract ID bị trùng, dừng gộp.")

combined_features_df = pd.concat(
    [combined_features_df, features_044_df],
    ignore_index=True,
    sort=False
).fillna(0)

combined_features_df = combined_features_df.loc[
    ~combined_features_df["contract_id"].astype(str).duplicated()
].reset_index(drop=True)

status_lookup_044 = status_044_df.set_index("contract_id")["status"].to_dict()

final_status_df["status"] = final_status_df.apply(
    lambda row: status_lookup_044.get(
        str(row["contract_id"]), row["status"]
    ),
    axis=1
)

print("\n=== SAU KHI GỘP ===")
print("Unique feature IDs:", combined_features_df["contract_id"].nunique())
print("Feature rows:", len(combined_features_df))
print("Unique status IDs:", final_status_df["contract_id"].nunique())
print(final_status_df["status"].value_counts().to_dict())

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

combined_features_df.to_csv(
    CHECKPOINT_DIR / "combined_features_41271.csv",
    index=False
)

final_status_df.to_csv(
    CHECKPOINT_DIR / "final_status_43232_v5.csv",
    index=False
)

print("\nĐã lưu checkpoint:")
print(CHECKPOINT_DIR / "combined_features_41271.csv")
print(CHECKPOINT_DIR / "final_status_43232_v5.csv")


Features 0.4.4: 12
ID bị trùng: set()

=== SAU KHI GỘP ===
Unique feature IDs: 41271
Feature rows: 41271
Unique status IDs: 43232
{'FEATURE_SUCCESS': 41271, 'NO_COMPILER_CANDIDATE': 1192, 'PROCESS_FAILED': 764, 'COMPILE_NO_RUNTIME': 5}

Đã lưu checkpoint:
/content/drive/MyDrive/IS6002/merged_dataset/opcode_features/checkpoints/combined_features_41271.csv
/content/drive/MyDrive/IS6002/merged_dataset/opcode_features/checkpoints/final_status_43232_v5.csv


In [ ]:

print("Shape:", combined_features_df.shape)
print("Số contract ID duy nhất:", combined_features_df["contract_id"].nunique())
print("Số ID bị trùng:", combined_features_df["contract_id"].duplicated().sum())

opcode_columns = [
    col for col in combined_features_df.columns
    if col != "contract_id"
]

print("Số cột opcode:", len(opcode_columns))
print("Giá trị thiếu:", int(combined_features_df[opcode_columns].isna().sum().sum()))
print("Giá trị âm:", int((combined_features_df[opcode_columns] < 0).sum().sum()))
print("Số giá trị vô hạn:", int(
    (~combined_features_df[opcode_columns].map(
        lambda x: pd.notna(x) and abs(float(x)) != float("inf")
    )).sum().sum()
))

display(combined_features_df.head())


Shape: (41271, 257)
Số contract ID duy nhất: 41271
Số ID bị trùng: 0
Số cột opcode: 256
Giá trị thiếu: 0
Giá trị âm: 0
Số giá trị vô hạn: 0


,contract_id,PUSH1,MSTORE,CALLDATASIZE,LT,PUSH2,JUMPI,CALLDATALOAD,PUSH29,SWAP1,...,UNKNOWN_0xD6,UNKNOWN_0xBE,UNKNOWN_0x28,UNKNOWN_0xB5,UNKNOWN_0xD1,UNKNOWN_0xEC,UNKNOWN_0xA9,UNKNOWN_0xAC,UNKNOWN_0x2D,UNKNOWN_0xCA
0,C000001,470,109,22.0,13.0,170.0,89.0,32.0,1.0,258.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,C000002,366,83,13.0,12.0,142.0,67.0,16.0,1.0,199.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,C000003,1081,218,45.0,62.0,455.0,243.0,64.0,0.0,571.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,C000004,1164,251,52.0,21.0,479.0,250.0,67.0,7.0,611.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,C000005,556,131,22.0,34.0,193.0,96.0,34.0,0.0,279.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
BASE_DIR = Path(
    "/content/drive/MyDrive/IS6002/merged_dataset/opcode_features"
)

folders = sorted(
    path for path in BASE_DIR.iterdir()
    if path.is_dir()
)

print("Tổng số folder:", len(folders))
print("\nDanh sách folder:")

for i, folder in enumerate(folders, start=1):
    print(f"{i}. {folder.name}")


Tổng số folder: 13

Danh sách folder:
1. checkpoints
2. feature_batches
3. feature_batches_manual_0410
4. feature_batches_retry
5. feature_batches_v2
6. legacy_044
7. legacy_046
8. legacy_048_test
9. legacy_049
10. status_batches
11. status_batches_manual_0410
12. status_batches_retry
13. status_batches_v2


In [ ]:

from pathlib import Path
import pandas as pd

file_path = Path(
    "/content/drive/MyDrive/IS6002/merged_dataset/"
    "opcode_features/checkpoints/combined_features_41271.csv"
)

print("File tồn tại:", file_path.exists())

if file_path.exists():
    df = pd.read_csv(file_path)

    print("Số contract:", len(df))
    print("Số cột:", len(df.columns))
    print("Kích thước bảng:", df.shape)

    display(df.head())


File tồn tại: True
Số contract: 41271
Số cột: 257
Kích thước bảng: (41271, 257)


,contract_id,PUSH1,MSTORE,CALLDATASIZE,LT,PUSH2,JUMPI,CALLDATALOAD,PUSH29,SWAP1,...,UNKNOWN_0xD6,UNKNOWN_0xBE,UNKNOWN_0x28,UNKNOWN_0xB5,UNKNOWN_0xD1,UNKNOWN_0xEC,UNKNOWN_0xA9,UNKNOWN_0xAC,UNKNOWN_0x2D,UNKNOWN_0xCA
0,C000001,470,109,22.0,13.0,170.0,89.0,32.0,1.0,258.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,C000002,366,83,13.0,12.0,142.0,67.0,16.0,1.0,199.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,C000003,1081,218,45.0,62.0,455.0,243.0,64.0,0.0,571.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,C000004,1164,251,52.0,21.0,479.0,250.0,67.0,7.0,611.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,C000005,556,131,22.0,34.0,193.0,96.0,34.0,0.0,279.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
import pandas as pd

FEATURES_PATH = "/content/drive/MyDrive/IS6002/merged_dataset/combined_features_41271.csv"

df = pd.read_csv(FEATURES_PATH)

unknown_columns = [
    col for col in df.columns
    if col.startswith("UNKNOWN_")
]

print("Số contract:", len(df))
print("Số cột UNKNOWN:", len(unknown_columns))

if unknown_columns:
    stats = pd.DataFrame({
        "Non_zero_count": (df[unknown_columns] != 0).sum(),
        "Total_sum": df[unknown_columns].sum(),
        "Max_value": df[unknown_columns].max()
    })

    stats["All_zero"] = stats["Non_zero_count"] == 0

    display(stats)

    print("\nSố cột UNKNOWN toàn số 0:",
          stats["All_zero"].sum())

    print("Số cột UNKNOWN có giá trị khác 0:",
          (~stats["All_zero"]).sum())
else:
    print("Không tìm thấy cột UNKNOWN.")

Số contract: 41271
Số cột UNKNOWN: 107


,Non_zero_count,Total_sum,Max_value,All_zero
UNKNOWN_0x4F,303,573.0,69.0,False
UNKNOWN_0x2E,234,367.0,35.0,False
UNKNOWN_0x4D,174,265.0,14.0,False
UNKNOWN_0x25,72,77.0,2.0,False
UNKNOWN_0x29,508,670.0,8.0,False
...,...,...,...,...
UNKNOWN_0xEC,79,906.0,820.0,False
UNKNOWN_0xA9,59,110.0,43.0,False
UNKNOWN_0xAC,61,146.0,79.0,False
UNKNOWN_0x2D,96,130.0,8.0,False



Số cột UNKNOWN toàn số 0: 0
Số cột UNKNOWN có giá trị khác 0: 107


In [155]:
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/IS6002/merged_dataset"

features_df = pd.read_csv(
    f"{BASE_DIR}/combined_features_41271.csv"
)

labels_df = pd.read_csv(
    f"{BASE_DIR}/labels.csv"
)

print("Feature columns:", features_df.columns.tolist())
print("Label columns:", labels_df.columns.tolist())

print("Số contract có feature:", features_df["contract_id"].nunique())
display(labels_df.head())

Feature columns: ['contract_id', 'PUSH1', 'MSTORE', 'CALLDATASIZE', 'LT', 'PUSH2', 'JUMPI', 'CALLDATALOAD', 'PUSH29', 'SWAP1', 'DIV', 'PUSH4', 'AND', 'DUP1', 'EQ', 'JUMPDEST', 'REVERT', 'CALLVALUE', 'ISZERO', 'POP', 'JUMP', 'MLOAD', 'ADD', 'DUP3', 'DUP2', 'SUB', 'DUP4', 'SWAP2', 'DUP5', 'EXP', 'NOT', 'SWAP3', 'RETURN', 'PUSH20', 'STOP', 'SLOAD', 'MUL', 'KECCAK256', 'GT', 'CALLER', 'DUP6', 'SSTORE', 'PUSH32', 'LOG3', 'DUP7', 'DUP8', 'SWAP4', 'TIMESTAMP', 'DUP9', 'CALL', 'RETURNDATASIZE', 'RETURNDATACOPY', 'CALLDATACOPY', 'SWAP5', 'INVALID', 'OR', 'LOG1', 'SHR', 'PUSH5', 'ADDRESS', 'SHL', 'SWAP6', 'EXTCODESIZE', 'GAS', 'DUP10', 'LOG2', 'DUP11', 'PUSH8', 'SELFBALANCE', 'STATICCALL', 'NUMBER', 'MOD', 'BLOCKHASH', 'SELFDESTRUCT', 'SWAP8', 'CODECOPY', 'PUSH31', 'PUSH16', 'PUSH15', 'PUSH21', 'PUSH6', 'PUSH9', 'DUP13', 'DUP12', 'SWAP12', 'DUP14', 'DUP15', 'SWAP7', 'PUSH3', 'MSTORE8', 'SWAP11', 'SWAP10', 'SWAP9', 'CODESIZE', 'PUSH14', 'PUSH7', 'UNKNOWN_0x4F', 'GASLIMIT', 'ORIGIN', 'GASPRICE', '

,contract_id,source_dataset,original_path,copied_path,vulnerability_type,label,sha256
0,C000001,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000001.sol,Overflow-Underflow,Vulnerable,b9b6f263036d2c4d1bfc8a39d3185ccb2a2d620ade2b90...
1,C000002,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000002.sol,Overflow-Underflow,Vulnerable,461671dfa9294ffe5bf9a4da92fbbc9173e45081adb956...
2,C000003,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000003.sol,Overflow-Underflow,Vulnerable,8043423d2d449063fbf182c76c44c79f2349524c34af47...
3,C000004,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000004.sol,Overflow-Underflow,Vulnerable,f5050f16b3daf9f3a061706e7dbb0eeff0da5bb22da51e...
4,C000005,SolidiFI,/home/bon/smartbugs-work/SolidiFI-benchmark/bu...,contracts/C000005.sol,Overflow-Underflow,Vulnerable,e9516787a0af1d5746c369092a48e7b92edc21f63762d0...


In [156]:
BASE_DIR = Path("/content/drive/MyDrive/IS6002/merged_dataset")

features_df = pd.read_csv(
    BASE_DIR / "combined_features_41271.csv"
)

labels_df = pd.read_csv(
    BASE_DIR / "labels.csv"
)

df = features_df.merge(
    labels_df[["contract_id", "label", "vulnerability_type", "source_dataset"]],
    on="contract_id",
    how="inner",
    validate="one_to_one"
)

print("Kích thước bảng sau khi ghép:", df.shape)
print("\nSố contract:", df["contract_id"].nunique())
print("\nPhân bố nhãn:")
print(df["label"].value_counts())

display(df.head())


Kích thước bảng sau khi ghép: (41271, 260)

Số contract: 41271

Phân bố nhãn:
label
Vulnerable        28128
Non-vulnerable    13143
Name: count, dtype: int64


,contract_id,PUSH1,MSTORE,CALLDATASIZE,LT,PUSH2,JUMPI,CALLDATALOAD,PUSH29,SWAP1,...,UNKNOWN_0xB5,UNKNOWN_0xD1,UNKNOWN_0xEC,UNKNOWN_0xA9,UNKNOWN_0xAC,UNKNOWN_0x2D,UNKNOWN_0xCA,label,vulnerability_type,source_dataset
0,C000001,470,109,22.0,13.0,170.0,89.0,32.0,1.0,258.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Vulnerable,Overflow-Underflow,SolidiFI
1,C000002,366,83,13.0,12.0,142.0,67.0,16.0,1.0,199.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Vulnerable,Overflow-Underflow,SolidiFI
2,C000003,1081,218,45.0,62.0,455.0,243.0,64.0,0.0,571.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Vulnerable,Overflow-Underflow,SolidiFI
3,C000004,1164,251,52.0,21.0,479.0,250.0,67.0,7.0,611.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Vulnerable,Overflow-Underflow,SolidiFI
4,C000005,556,131,22.0,34.0,193.0,96.0,34.0,0.0,279.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Vulnerable,Overflow-Underflow,SolidiFI
